# Erodabilidad y pérdidas de suelo por erosión hídrica — Cuenca del Río Amaime
## Modelo USLE/RUSLE · Versión 6.0

Flujo de trabajo reproducible para estimar la **erodabilidad de los suelos** y las
**pérdidas de suelo por erosión hídrica** en la cuenca del río Amaime, aplicando la
**Ecuación Universal de Pérdida de Suelo Revisada (USLE/RUSLE)**:

$$A = R \cdot K \cdot LS \cdot C \cdot P$$

### Objetivos

**General.** Determinar la erodabilidad de los suelos de la cuenca del río Amaime
como estrategia para su conservación.

| # | Objetivo específico | Sección |
|---|---|---|
| I | Caracterizar las propiedades físicas y químicas de los suelos | 19 |
| II | Establecer la erodabilidad mediante el factor **K** y su relación con dichas propiedades | 20 |
| III | Evaluar el riesgo potencial de erosión hídrica mediante el índice **LS × R** | 24 |
| IV | Estimar las pérdidas de suelo y su distribución espacial (USLE/RUSLE) | 25 |
| V | Modelar la variación temporal de las pérdidas 2010–2025 actualizando el factor **R** | 26 |

### Fuentes e insumos

| Factor | Fuente | Método |
|---|---|---|
| **LS** (topográfico) | DEM Copernicus GLO‑30 (`COPERNICUS/DEM/GLO30_2024_1`, Earth Engine) | Mitasova & Mitas (2001) sobre acumulación de flujo D8 (WhiteboxTools) |
| **R** (erosividad) | CHIRPS diario (`UCSB-CHG/CHIRPS/DAILY`, Earth Engine) | Índice de Fournier Modificado (MFI) → R (Renard & Freimund, 1994) |
| **K** (erodabilidad) | Base de suelos de la tesis (`Cuenca Amaime Tesis Dayana.xlsx`) | Wischmeier & Smith (1978), unidades SI |
| **C** (cobertura) | Base de suelos + ESA WorldCover v200 (Earth Engine) | Factor de cultivos por punto; tabla por cobertura para el ráster |
| **P** (prácticas) | Base de suelos (capa CVC) | Factor P por punto; escenario base P = 1 para el ráster |

Los insumos raster se **descargan automáticamente desde la API de Google Earth
Engine**; no se requiere descarga manual. Todos los resultados se calculan tanto en
los **117 puntos de muestreo** como en **rásteres continuos a 30 m** recortados a
la zona de estudio.


In [ ]:
# (Opcional) Montaje de Google Drive si el notebook se ejecuta en Google Colab.
# En entorno local no realiza ninguna acción.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Entorno local detectado (no Colab): se omite el montaje de Google Drive.')


## 1. Configuración del proyecto

Se centralizan **todas** las dependencias, rutas y parámetros globales. Ningún
otro bloque del notebook redefine estas constantes.


In [ ]:
# Instalación de dependencias (idempotente). En Colab la mayoría ya están presentes.
%pip install --quiet earthengine-api rasterio whitebox pandas geopandas openpyxl alphashape folium branca matplotlib


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

import ee
import whitebox

warnings.filterwarnings('ignore')

try:
    from IPython.display import display
except ImportError:  # ejecución fuera de Jupyter/IPython
    display = print

print('Librerías importadas.')


In [ ]:
# =====================================================================
#  PARÁMETROS GLOBALES DEL PROYECTO RUSLE — CUENCA DEL RÍO AMAIME
# =====================================================================

# --- Directorios (todo el proyecto vive bajo DIR_BASE) ---------------
DIR_BASE    = Path(r"D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime")
DIR_ENTRADA = DIR_BASE / 'Entrada'
DIR_SALIDA  = DIR_BASE / 'Salida'
for d in (DIR_ENTRADA, DIR_SALIDA):
    d.mkdir(parents=True, exist_ok=True)

# --- Subcarpetas de Salida por tipo de producto ---
DIR_RASTERS = DIR_SALIDA / 'rasters'    # .tif
DIR_TABLAS  = DIR_SALIDA / 'tablas'     # .xlsx, .gpkg, .csv
DIR_MAPAS   = DIR_SALIDA / 'mapas'      # .html
DIR_FIGURAS = DIR_SALIDA / 'figuras'    # .png
for d in (DIR_RASTERS, DIR_TABLAS, DIR_MAPAS, DIR_FIGURAS):
    d.mkdir(parents=True, exist_ok=True)

# --- Insumos: tabla de puntos y base de suelos de la tesis ---------
ARCHIVO_PUNTOS = DIR_ENTRADA / 'Cuenca amaime.xlsx'
ARCHIVO_SUELOS = DIR_ENTRADA / 'Cuenca Amaime Tesis Dayana.xlsx'

# --- Rasters de insumo descargados desde Earth Engine --------------
DEM_FILE            = DIR_RASTERS / 'DEM_Amaime_Copernicus30m.tif'
CHIRPS_MENSUAL_FILE = DIR_RASTERS / 'CHIRPS_Mensual_Climatologia_Amaime_2020_2025.tif'
CHIRPS_MENSUAL_CORR_FILE = DIR_RASTERS / 'CHIRPS_Mensual_Climatologia_2010_2025.tif'

# --- Rasters derivados (hidrología / geomorfometría) ---------------
DEM_BREACH_FILE        = DIR_RASTERS / 'DEM_Amaime_Breach.tif'
D8_POINTER_FILE        = DIR_RASTERS / 'D8_Pointer.tif'
FLOW_ACC_FILE          = DIR_RASTERS / 'Flow_Accumulation.tif'
FLOW_LENGTH_FILE       = DIR_RASTERS / 'Longitud_Ladera.tif'          # distancia de flujo a la salida
STREAMS_FILE           = DIR_RASTERS / 'Streams.tif'
FLOW_LENGTH_RUSLE_FILE = DIR_RASTERS / 'Longitud_Ladera_RUSLE.tif'    # distancia al primer cauce
SLOPE_DEG_FILE         = DIR_RASTERS / 'Pendiente_grados.tif'
LS_FILE                = DIR_RASTERS / 'LS_Mitasova_2001.tif'
ZONA_INFLUENCIA        = DIR_TABLAS / 'Zona_Influencia.gpkg'

# --- Productos finales --------------------------------------------
SALIDA_PUNTOS_XLSX = DIR_TABLAS / 'Cuenca_amaime_completo.xlsx'
SALIDA_PUNTOS_GPKG = DIR_TABLAS / 'Cuenca_amaime_completo.gpkg'

# --- Sistemas de referencia --------------------------------------
CRS_PROYECTADO = 'EPSG:3115'   # MAGNA-SIRGAS / Colombia Bogotá — cálculos métricos
CRS_GEOGRAFICO = 'EPSG:4326'   # WGS84 — muestreo de CHIRPS y AOI de Earth Engine

# --- Nombres de columnas en la tabla de puntos ------------------
COL_X_PROJ, COL_Y_PROJ = 'COOR_X', 'COOR_Y'   # coordenadas proyectadas (EPSG:3115)
COL_LON,   COL_LAT     = 'LONG',   'LAT'      # coordenadas geográficas (EPSG:4326)

# --- Parámetros de malla y procesamiento hidrológico ----------
CELLSIZE_M         = 30.0    # resolución de trabajo (m)
STREAM_THRESHOLD   = 50      # umbral de acumulación de flujo para definir cauces (celdas)
UMBRAL_RED_DRENAJE = STREAM_THRESHOLD          # alias usado en los diagnósticos
BREACH_DIST        = 100     # distancia máx. de breaching de depresiones (celdas)

# --- Exponentes del factor LS de Mitasova & Mitas (2001) ------
LS_M_EXP = 0.4
LS_N_EXP = 1.3
LS_UMBRALES_EXTREMOS = [50, 75, 100]           # umbrales para el diagnóstico de LS extremos

# --- Zona de influencia (envolvente cóncava + buffer) --------
ALPHA_CONCAVE_HULL = 0.0001   # alpha para la envolvente cóncava (unidades 1/m en EPSG:3115)
BUFFER_ZONA_M      = 5000     # margen de la zona de influencia alrededor de los puntos (m)

# --- Google Earth Engine ---------------------------------------
EE_PROJECT         = 'ee-pracagro2'   # proyecto de Google Cloud habilitado para Earth Engine
FORZAR_DESCARGA_EE = False            # True = vuelve a descargar DEM y CHIRPS aunque ya existan
MARGEN_AOI_M       = 5000             # margen alrededor de los puntos para el AOI (m); holgado
                                     # para dar contexto de cuenca a la acumulación de flujo D8
CHIRPS_ANIO_INI, CHIRPS_ANIO_FIN = 2020, 2025    # periodo del Factor R del estudio (reproduce Dayana)
CORR_ANIO_INI, CORR_ANIO_FIN     = 2010, 2025    # periodo para el análisis de correlación P mensual ↔ R

print('Configuración cargada.')
print(f'  DIR_BASE         : {DIR_BASE}')
print(f'  Umbral de cauces : {STREAM_THRESHOLD} celdas')
print(f'  Resolución       : {CELLSIZE_M:.0f} m  |  CRS métrico: {CRS_PROYECTADO}')
print(f'  Proyecto EE      : {EE_PROJECT}')


## 2. Autenticación e inicialización de Google Earth Engine

La primera vez que se ejecuta en un entorno nuevo, `ee.Authenticate()` abre el
navegador para autorizar el acceso. En ejecuciones posteriores basta con
`ee.Initialize()`. Se usa el proyecto de Google Cloud definido en `EE_PROJECT`.


In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
    print(f'Earth Engine inicializado (proyecto: {EE_PROJECT}).')
except Exception:
    # Primera ejecución en este entorno: flujo de autenticación interactivo.
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f'Earth Engine autenticado e inicializado (proyecto: {EE_PROJECT}).')


## 3. Funciones del flujo de trabajo

Todas las funciones reutilizables se agrupan aquí, organizadas por categoría:
**E/S raster**, **descarga desde Earth Engine**, **SIG y extracción a puntos**,
**hidrología / geomorfometría**, **climatología**, **erodabilidad**,
**control de calidad** y **exportación**. Ningún otro bloque del notebook
redefine lógica ya presente en esta sección.


### 3.1 Gestión de archivos y E/S raster


In [ ]:
def leer_raster(ruta_raster, banda=1):
    """Lee una banda de un raster y devuelve (datos float32 con NaN, perfil, nodata)."""
    with rasterio.open(ruta_raster) as src:
        datos = src.read(banda).astype('float32')
        perfil = src.profile.copy()
        datos = np.where(datos == src.nodata, np.nan, datos)
    return datos, perfil, src.nodata


def guardar_raster(ruta_salida, datos, perfil_base, nodata=np.nan):
    """Escribe un array 2D como GeoTIFF float32 de una sola banda."""
    perfil = perfil_base.copy()
    perfil.update(dtype='float32', count=1, nodata=nodata)
    with rasterio.open(ruta_salida, 'w', **perfil) as dst:
        dst.write(datos.astype('float32'), 1)


### 3.2 Descarga de insumos desde Earth Engine

> **Nota (DEM):** se usa `COPERNICUS/DEM/GLO30_2024_1`, la colección vigente que
> reemplazó a la ya obsoleta `COPERNICUS/DEM/GLO30`. Es el mismo DEM Copernicus
> GLO‑30; solo cambia la ingesta en Earth Engine.

`_ee_descargar_geotiff()` resuelve una `ee.Image` a un GeoTIFF local mediante
`getDownloadURL` (válido para áreas pequeñas como esta cuenca; para AOIs grandes
usar `ee.batch.Export.image.toDrive`). Sobre esa utilidad se construyen las
descargas del **DEM Copernicus GLO‑30** y de la **climatología mensual CHIRPS**.


In [ ]:
def _ee_descargar_geotiff(imagen, region, ruta_salida, scale, crs, nombre='imagen'):
    """Descarga una ee.Image como GeoTIFF local.

    region : ee.Geometry con la extensión a exportar.
    scale  : resolución de salida en metros.
    crs    : CRS de salida (p. ej. 'EPSG:3115').
    """
    import urllib.request, shutil, zipfile, tempfile

    ruta_salida = Path(ruta_salida)
    ruta_salida.parent.mkdir(parents=True, exist_ok=True)

    url = imagen.getDownloadURL({
        'region': region, 'scale': scale, 'crs': crs, 'format': 'GEO_TIFF',
    })
    print(f'  Descargando {nombre} desde Earth Engine ...')

    tmp = Path(tempfile.gettempdir()) / (ruta_salida.stem + '_ee_tmp')
    urllib.request.urlretrieve(url, tmp)

    # getDownloadURL puede devolver el GeoTIFF directo o un ZIP que lo contiene.
    if zipfile.is_zipfile(tmp):
        with zipfile.ZipFile(tmp) as z:
            tif = [n for n in z.namelist() if n.lower().endswith('.tif')][0]
            with z.open(tif) as src, open(ruta_salida, 'wb') as dst:
                shutil.copyfileobj(src, dst)
        tmp.unlink()
    else:
        # copyfile (no move) para funcionar aunque el temporal y el destino
        # estén en unidades de disco distintas (p. ej. C: -> D:)
        shutil.copyfile(tmp, ruta_salida)
        tmp.unlink()

    # Re-codificar a compresión LZW: Earth Engine entrega GeoTIFF con una
    # compresión (código 32946) que WhiteboxTools no sabe leer.
    with rasterio.open(ruta_salida) as s:
        perfil = s.profile.copy()
        datos = s.read()
    perfil.update(compress='lzw', predictor=1, tiled=False, BIGTIFF='IF_SAFER')
    perfil.pop('interleave', None)
    with rasterio.open(ruta_salida, 'w', **perfil) as d:
        d.write(datos)

    with rasterio.open(ruta_salida) as s:
        print(f'  OK: {ruta_salida.name} | CRS {s.crs} | {s.width}x{s.height} px | '
              f'{s.count} banda(s) | compresión {s.profile.get("compress")}')
    return ruta_salida


def descargar_dem_copernicus(region, ruta_salida, scale=30, crs=CRS_PROYECTADO):
    """DEM Copernicus GLO‑30 (COPERNICUS/DEM/GLO30): mosaico recortado al AOI."""
    dem = (ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1')
             .select('DEM')
             .filterBounds(region)
             .mosaic()
             .clip(region))
    return _ee_descargar_geotiff(dem, region, ruta_salida, scale, crs,
                                 nombre='DEM Copernicus GLO-30')


def construir_chirps_climatologia_mensual(region, anio_ini=2020, anio_fin=2025):
    """Imagen de 12 bandas (P_01..P_12) con la climatología mensual de lluvia CHIRPS.

    Para cada mes se calcula el acumulado mensual de cada año del período y luego
    el promedio interanual (mm/mes).
    """
    chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .filterDate(f'{anio_ini}-01-01', f'{anio_fin + 1}-01-01')
                .filterBounds(region)
                .select('precipitation'))
    anios = ee.List.sequence(anio_ini, anio_fin)

    def clim_mes(m):
        m = ee.Number(m)

        def acum_anual(y):
            y = ee.Number(y)
            return (chirps
                    .filter(ee.Filter.calendarRange(y, y, 'year'))
                    .filter(ee.Filter.calendarRange(m, m, 'month'))
                    .sum())

        acum_por_anio = ee.ImageCollection(anios.map(acum_anual))
        return acum_por_anio.mean().rename(ee.String('P_').cat(m.format('%02d')))

    bandas = ee.List.sequence(1, 12).map(clim_mes)
    return (ee.ImageCollection(bandas).toBands()
            .rename([f'P_{i:02d}' for i in range(1, 13)])
            .clip(region))


def descargar_chirps_mensual(region, ruta_salida, anio_ini=2020, anio_fin=2025,
                             scale=5566, crs=CRS_GEOGRAFICO):
    """Descarga la climatología mensual CHIRPS (12 bandas) al AOI. scale≈0.05° nativo."""
    img = construir_chirps_climatologia_mensual(region, anio_ini, anio_fin)
    return _ee_descargar_geotiff(img, region, ruta_salida, scale, crs,
                                 nombre=f'CHIRPS climatología mensual {anio_ini}-{anio_fin}')


### 3.3 SIG: geometría y extracción a puntos


In [ ]:
def leer_tabla_puntos(ruta_excel):
    """Lee la tabla de puntos de muestreo (Excel) y devuelve un DataFrame."""
    return pd.read_excel(ruta_excel)


def construir_geodataframe(df, col_x, col_y, crs_salida):
    """Convierte un DataFrame con coordenadas (col_x, col_y) en un GeoDataFrame de puntos."""
    return gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[col_x], df[col_y]), crs=crs_salida
    )


def extraer_valor_raster_puntos(gdf, ruta_raster, col_name, banda=1):
    """Muestrea `ruta_raster` (banda dada) en cada punto y lo guarda en `col_name`.

    Reproyecta automáticamente los puntos al CRS del raster antes de muestrear.
    """
    with rasterio.open(ruta_raster) as src:
        puntos_reproj = gdf.to_crs(src.crs)
        coords = list(zip(puntos_reproj.geometry.x, puntos_reproj.geometry.y))
        valores = [v[0] if v[0] != src.nodata else np.nan
                   for v in src.sample(coords, indexes=banda)]
    gdf[col_name] = valores
    return gdf


### 3.4 Hidrología y geomorfometría

- **Pendiente (grados)** a partir del gradiente del DEM.
- **Factor LS — Mitasova & Mitas (2001):**
  $LS = (A_s/22.13)^{m}\,(\sin\theta/0.0896)^{n}$, con $m=0.4$, $n=1.3$ y
  $A_s \approx$ (acumulación de flujo D8) $\times$ (tamaño de celda).
- **Extracción unificada** de variables topográficas a los puntos, con
  `Longitud_Ladera_m` como única longitud oficial y piso físico de un tamaño de celda.
- **Factores L y S clásicos** (Wischmeier & Smith, 1978; McCool et al., 1987).


In [ ]:
def calcular_pendiente_grados(dem, pixel_size):
    """Pendiente en grados: arctan(|∇z|) · 180/π, con ∇z estimado por diferencias centradas."""
    dy, dx = np.gradient(dem, pixel_size)
    slope_rad = np.arctan(np.sqrt(dx ** 2 + dy ** 2))
    return np.degrees(slope_rad)


def pendiente_grados_a_porcentaje(slope_deg):
    """Convierte pendiente de grados a porcentaje: tan(θ) · 100."""
    return np.tan(np.radians(slope_deg)) * 100.0


def calcular_factor_ls_mitasova(flow_accum, slope_deg, cellsize, m=0.4, n=1.3):
    """Factor topográfico LS de Mitasova & Mitas (2001), basado en área de captación.

        As = flow_accum · cellsize
        LS = (As / 22.13)^m · (sin θ / 0.0896)^n
    """
    slope_rad = np.radians(slope_deg)
    As = flow_accum * cellsize
    As = np.where(As <= 0, np.nan, As)
    LS = ((As / 22.13) ** m) * ((np.sin(slope_rad) / 0.0896) ** n)
    return np.where(np.isinf(LS), np.nan, LS)


def agregar_variables_topograficas_unificada(puntos_gdf, ls_mitasova_raster, slope_deg_raster,
                                             flow_acc_raster, flow_length_oficial_raster):
    """Extrae las variables topográficas a los puntos consolidando 'Longitud_Ladera_m'
    como la única longitud oficial. Ningún punto (ladera o cauce) queda con longitud 0:
    se aplica un piso igual al tamaño de celda (CELLSIZE_M).
    """
    puntos = puntos_gdf.copy()

    # 1. Variables base
    puntos = extraer_valor_raster_puntos(puntos, ls_mitasova_raster, 'LS_Mitasova')
    puntos = extraer_valor_raster_puntos(puntos, slope_deg_raster, 'Pendiente_grados')
    puntos['Pendiente_pct'] = pendiente_grados_a_porcentaje(puntos['Pendiente_grados'])
    puntos = extraer_valor_raster_puntos(puntos, flow_acc_raster, 'FlowAccum')

    # 2. Longitud de ladera oficial (distancia D8 al primer cauce)
    puntos = extraer_valor_raster_puntos(puntos, flow_length_oficial_raster, 'Longitud_Ladera_m')

    # 3. Tratamiento de ceros y nulos: piso de un tamaño de celda para todos los registros
    puntos['Longitud_Ladera_m'] = puntos['Longitud_Ladera_m'].replace(0, np.nan).fillna(CELLSIZE_M)
    puntos.loc[puntos['Longitud_Ladera_m'] < CELLSIZE_M, 'Longitud_Ladera_m'] = CELLSIZE_M

    return puntos


def calcular_factores_ls_clasicos(puntos_gdf, cellsize=30.0):
    """Factores L y S clásicos y LS clásico (Wischmeier & Smith, 1978; McCool et al., 1987)."""
    df = puntos_gdf.copy()

    # Pendiente en radianes y seno de la pendiente
    theta = np.radians(df['Pendiente_grados'])
    sin_theta = np.sin(theta)

    # Factor S (McCool et al., 1987)
    df['Factor_S'] = np.where(
        df['Pendiente_pct'] < 9,
        10.8 * sin_theta + 0.03,
        16.8 * sin_theta - 0.50,
    )

    # Factor L (Wischmeier & Smith, 1978): L = (lambda / 22.13) ** m
    s = df['Pendiente_pct']
    m = np.where(s >= 5, 0.5,
                 np.where(s >= 3, 0.4,
                          np.where(s >= 1, 0.3, 0.2)))
    df['m_param'] = m

    # Longitud de ladera sin truncar en 400 m (cálculo continuo solicitado)
    lambda_val = df['Longitud_Ladera_m']
    df['Factor_L'] = (lambda_val / 22.13) ** m

    # LS clásico: producto directo, sin truncamientos
    df['LS_Clasico'] = df['Factor_L'] * df['Factor_S']
    return df


### 3.5 Climatología: Índice de Fournier Modificado y Factor R

$$MFI = \frac{\sum_{i=1}^{12} P_i^{2}}{P_{anual}}
\qquad
R = \begin{cases}
0.7397\,(MFI)^{1.847} & MFI < 55\\[4pt]
95.77 - 6.081\,MFI + 0.4770\,MFI^{2} & MFI \ge 55
\end{cases}$$


In [ ]:
def calcular_mfi_y_r(df_mensual):
    """Índice de Fournier Modificado (MFI) y Factor R a partir de 12 columnas P_01..P_12.

    MFI = sum(Pi^2) / P_anual
    R = 0.7397 * MFI^1.847                     si MFI < 55
    R = 95.77 - 6.081*MFI + 0.4770*MFI^2       si MFI >= 55
    """
    cols_p = [f'P_{i:02d}' for i in range(1, 13)]
    p_anual = df_mensual[cols_p].sum(axis=1)
    sum_p2 = (df_mensual[cols_p] ** 2).sum(axis=1)

    mfi = sum_p2 / p_anual
    factor_r = pd.Series(
        np.where(mfi < 55,
                 0.7397 * (mfi ** 1.847),
                 95.77 - 6.081 * mfi + 0.4770 * (mfi ** 2)),
        index=mfi.index,
    )
    return p_anual, mfi, factor_r


### 3.6 Erodabilidad del suelo: Factor K (Wischmeier & Smith, 1978)

$$K=\frac{2.1\times10^{-4}\,(12-MO)\,M^{1.14} + 3.25\,(s-2) + 2.5\,(p-3)}{100}
\qquad
M=(\%Limo+\%Arena\ muy\ fina)\,(100-\%Arcilla)$$

$MO$ = % de materia orgánica, $s$ = código de estructura (1–4), $p$ = código de
permeabilidad (1–6). La función queda lista; el Factor K se calcula en la
Sección 12 **solo si** la tabla de puntos incluye las columnas edáficas.


In [ ]:
def calcular_factor_k_wischmeier(om_pct, limo_pct, arena_muy_fina_pct, arcilla_pct,
                                 codigo_estructura, codigo_permeabilidad, convertir_si=False):
    """Factor K de erodabilidad del suelo — ecuación del nomograma de Wischmeier & Smith (1978).

        M = (%limo + %arena muy fina) * (100 - %arcilla)
        K = [ 2.1e-4 * M**1.14 * (12 - MO) + 3.25*(s - 2) + 2.5*(p - 3) ] / 100

    Resultado en unidades inglesas; con convertir_si=True se multiplica por 0.1317
    para obtener unidades SI (t·ha·h·(ha·MJ·mm)^-1).
    """
    M = (limo_pct + arena_muy_fina_pct) * (100.0 - arcilla_pct)
    K = (2.1e-4 * (M ** 1.14) * (12.0 - om_pct)
         + 3.25 * (codigo_estructura - 2.0)
         + 2.5 * (codigo_permeabilidad - 3.0)) / 100.0
    return K * 0.1317 if convertir_si else K


### 3.7 Control de calidad


In [ ]:
def detectar_valores_extremos(serie, nombre, factor_iqr=3.0):
    """Identifica valores atípicos extremos con el criterio del rango intercuartílico (IQR).

    Un valor se marca como extremo si cae fuera de [Q1 - k·IQR, Q3 + k·IQR].
    Se usa k=3.0 (en vez del 1.5 clásico) para señalar solo atípicos extremos.
    """
    s = pd.Series(serie).dropna()
    if s.empty:
        return pd.DataFrame()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - factor_iqr * iqr, q3 + factor_iqr * iqr
    atipicos = s[(s < lim_inf) | (s > lim_sup)]
    print(f"[QC] {nombre}: {len(atipicos)} valores extremos de {len(s)} "
          f"(límites IQR x{factor_iqr}: [{lim_inf:.2f}, {lim_sup:.2f}])")
    return atipicos


def reporte_nodata(datos, nombre):
    """Reporta el porcentaje de píxeles nodata/NaN de un raster."""
    total = datos.size
    n_nan = np.isnan(datos).sum()
    pct = 100 * n_nan / total if total else np.nan
    print(f"[QC] {nombre}: {n_nan:,} px nodata de {total:,} ({pct:.2f}%)")
    return pct


def verificar_rango_fisico(valor_min, valor_max, limite_inf, limite_sup, nombre):
    """Verifica si el rango observado es físicamente plausible y avisa si no lo es."""
    ok = (valor_min >= limite_inf) and (valor_max <= limite_sup)
    estado = "OK" if ok else "FUERA DE RANGO ESPERADO"
    print(f"[QC] {nombre}: observado [{valor_min:.3f}, {valor_max:.3f}] vs. "
          f"esperado [{limite_inf}, {limite_sup}] -> {estado}")
    return ok


def resumen_percentiles_completo(serie, nombre):
    """Resumen estadístico completo para variables críticas de QC: mínimo, máximo,
    media, mediana, desviación estándar y percentiles P50/P75/P90/P95/P99.
    Devuelve un dict de una fila, apto para construir tablas comparativas.
    """
    s = pd.Series(serie).dropna().astype('float64')
    if s.empty:
        return {"variable": nombre, "n": 0}
    return {
        "variable": nombre,
        "n": int(s.size),
        "min": s.min(),
        "max": s.max(),
        "media": s.mean(),
        "mediana": s.median(),
        "desv_std": s.std(),
        "P50": s.quantile(0.50),
        "P75": s.quantile(0.75),
        "P90": s.quantile(0.90),
        "P95": s.quantile(0.95),
        "P99": s.quantile(0.99),
    }


def diagnostico_valores_extremos_ls(LS, flow_acc=None, umbral_red=None, umbrales=(50, 75, 100)):
    """Cuantifica qué fracción de píxeles de LS supera distintos umbrales extremos y,
    si se entrega `flow_acc` y `umbral_red`, qué proporción de esos píxeles coincide
    con celdas de cauce (acumulación de flujo alta). Permite distinguir laderas reales
    muy largas/empinadas de artefactos numéricos de la fórmula de área de contribución
    en celdas de cauce. No modifica los datos: es solo diagnóstico.
    """
    datos = LS[~np.isnan(LS)]
    n_total = datos.size
    filas = []
    for u in umbrales:
        mascara = LS > u
        n_extremos = int(np.nansum(mascara))
        pct = 100 * n_extremos / n_total if n_total else np.nan
        fila = {"umbral_LS": f"> {u}", "n_pixeles": n_extremos, "pct_del_total": pct}
        if flow_acc is not None and umbral_red is not None:
            con_dato = mascara & ~np.isnan(flow_acc)
            n_con_dato = int(np.nansum(con_dato))
            if n_con_dato:
                n_en_cauce = int(np.nansum(con_dato & (flow_acc > umbral_red)))
                fila["pct_coincide_con_cauce"] = 100 * n_en_cauce / n_con_dato
            else:
                fila["pct_coincide_con_cauce"] = np.nan
        filas.append(fila)
    tabla = pd.DataFrame(filas)
    print(f"[QC] Factor LS — diagnóstico de valores extremos (n total válido = {n_total:,}):")
    print(tabla.to_string(index=False))
    return tabla


### 3.8 Exportación


In [ ]:
def exportar_resultados(puntos_gdf, ruta_xlsx=None, ruta_gpkg=None, ruta_csv=None):
    """Exporta la tabla de puntos con variables RUSLE a Excel, GeoPackage y/o CSV."""
    rutas_generadas = []

    if ruta_xlsx is not None:
        ruta_xlsx = Path(ruta_xlsx)
        ruta_xlsx.parent.mkdir(parents=True, exist_ok=True)
        puntos_gdf.drop(columns='geometry', errors='ignore').to_excel(ruta_xlsx, index=False)
        rutas_generadas.append(ruta_xlsx)

    if ruta_csv is not None:
        ruta_csv = Path(ruta_csv)
        ruta_csv.parent.mkdir(parents=True, exist_ok=True)
        puntos_gdf.drop(columns='geometry', errors='ignore').to_csv(ruta_csv, index=False)
        rutas_generadas.append(ruta_csv)

    if ruta_gpkg is not None:
        ruta_gpkg = Path(ruta_gpkg)
        ruta_gpkg.parent.mkdir(parents=True, exist_ok=True)
        puntos_gdf.to_file(ruta_gpkg, driver='GPKG')
        rutas_generadas.append(ruta_gpkg)

    for r in rutas_generadas:
        print(f'  Exportado: {r}')
    return rutas_generadas


print('Funciones cargadas: E/S raster, Earth Engine, SIG, hidrología, climatología, '
      'erodabilidad, control de calidad y exportación.')


## 4. Área de interés (AOI)

Conforme al método del estudio, el área de análisis es el **buffer de 5 km
alrededor de los 117 puntos de muestreo**. El AOI de descarga de Earth Engine es
el rectángulo envolvente de los puntos ampliado `MARGEN_AOI_M` = 5 000 m, en
coordenadas geográficas (WGS84).


In [ ]:
# La tabla se relee formalmente en la Sección 6; aquí solo se usa para el AOI.
_df_aoi = leer_tabla_puntos(ARCHIVO_PUNTOS)

lon_min, lon_max = float(_df_aoi[COL_LON].min()), float(_df_aoi[COL_LON].max())
lat_min, lat_max = float(_df_aoi[COL_LAT].min()), float(_df_aoi[COL_LAT].max())

# Margen en grados (1° ≈ 111.32 km) a partir de MARGEN_AOI_M (buffer de 5 km del método)
margen_deg = MARGEN_AOI_M / 111_320.0

AOI = ee.Geometry.Rectangle([
    lon_min - margen_deg, lat_min - margen_deg,
    lon_max + margen_deg, lat_max + margen_deg,
])

print('=== ÁREA DE INTERÉS (WGS84) — envolvente de los 117 puntos + buffer 5 km ===')
print(f'  Longitud : [{lon_min - margen_deg:.4f}, {lon_max + margen_deg:.4f}]')
print(f'  Latitud  : [{lat_min - margen_deg:.4f}, {lat_max + margen_deg:.4f}]')
print(f'  Margen   : {MARGEN_AOI_M:.0f} m  (~{margen_deg:.4f}°)')
print(f'  Área     : {AOI.area().getInfo() / 1e6:,.1f} km²')


## 5. Descarga de insumos desde Earth Engine

Se descargan a `Salida/` únicamente si el archivo no existe todavía (o si
`FORZAR_DESCARGA_EE = True`). Así el notebook es reejecutable sin volver a
descargar en cada corrida.


### 5.1 DEM Copernicus GLO‑30


In [ ]:
if FORZAR_DESCARGA_EE or not DEM_FILE.exists():
    descargar_dem_copernicus(AOI, DEM_FILE, scale=int(CELLSIZE_M), crs=CRS_PROYECTADO)
else:
    print(f'DEM ya presente, se omite descarga: {DEM_FILE.name}')


### 5.2 CHIRPS — climatología mensual 2020–2025


In [ ]:
if FORZAR_DESCARGA_EE or not CHIRPS_MENSUAL_FILE.exists():
    descargar_chirps_mensual(AOI, CHIRPS_MENSUAL_FILE,
                             anio_ini=CHIRPS_ANIO_INI, anio_fin=CHIRPS_ANIO_FIN,
                             crs=CRS_GEOGRAFICO)
else:
    print(f'CHIRPS ya presente, se omite descarga: {CHIRPS_MENSUAL_FILE.name}')


## 6. Tabla de puntos y área de estudio

El **área de estudio** es el **buffer de 5 km alrededor de los 117 puntos de
muestreo** (`BUFFER_ZONA_M = 5000`), conforme al método del estudio. Se genera a
partir de la envolvente de los puntos (cóncava si es válida, si no convexa) y se
guarda en `Salida/Zona_Influencia.gpkg`.


In [ ]:
# 6.1 Tabla de puntos de muestreo
df_puntos = leer_tabla_puntos(ARCHIVO_PUNTOS)

print('Columnas encontradas:')
print(df_puntos.columns.tolist())
print(f'\nRegistros: {len(df_puntos)}')
display(df_puntos.head())


In [ ]:
# 6.2 Área de estudio = buffer de 5 km de los puntos (máscara de todo el modelo).
#     Se (re)genera si el archivo no existe o si contiene una geometría inválida.
def _zona_gpkg_valida(ruta):
    try:
        g = gpd.read_file(ruta)
        return (len(g) > 0 and g.geometry.notna().all()
                and g.geometry.is_valid.all() and float(g.geometry.area.sum()) > 0)
    except Exception:
        return False


if ZONA_INFLUENCIA.exists() and _zona_gpkg_valida(ZONA_INFLUENCIA):
    zona_gdf = gpd.read_file(ZONA_INFLUENCIA)
    print(f'Área de estudio cargada desde disco: {ZONA_INFLUENCIA.name}')
else:
    pts = gpd.GeoSeries(gpd.points_from_xy(df_puntos[COL_X_PROJ], df_puntos[COL_Y_PROJ]),
                        crs=CRS_PROYECTADO)
    envolvente_convexa = pts.unary_union.convex_hull
    envolvente = None
    if ALPHA_CONCAVE_HULL:
        try:
            import alphashape
            cand = alphashape.alphashape(list(zip(pts.x, pts.y)), ALPHA_CONCAVE_HULL)
            # se acepta solo si es un polígono válido y cubre buena parte de la envolvente convexa
            if (cand is not None and not cand.is_empty and cand.is_valid
                    and cand.area >= 0.4 * envolvente_convexa.area):
                envolvente = cand.buffer(0)
        except Exception as e:
            print('alphashape no disponible o falló; se usa envolvente convexa:', e)
    if envolvente is None:
        envolvente = envolvente_convexa
        print('Envolvente cóncava no válida -> se usa envolvente CONVEXA de los puntos.')

    geom_buffer = envolvente.buffer(BUFFER_ZONA_M)
    zona_gdf = gpd.GeoDataFrame({'ID': [1]}, geometry=[geom_buffer], crs=CRS_PROYECTADO)
    ZONA_INFLUENCIA.parent.mkdir(parents=True, exist_ok=True)
    if ZONA_INFLUENCIA.exists():
        ZONA_INFLUENCIA.unlink()
    zona_gdf.to_file(ZONA_INFLUENCIA)
    print(f'Zona de influencia (re)generada: {ZONA_INFLUENCIA.name}  |  '
          f'Área: {zona_gdf.area.iloc[0] / 10000:,.0f} ha')


## 7. Procesamiento hidrológico — WhiteboxTools

Cadena D8 estándar sobre el DEM descargado:

DEM → *breaching* de depresiones → dirección de flujo (D8 pointer) →
acumulación de flujo → longitud de flujo a la salida → red de drenaje
(umbral = 50 celdas) → **longitud de ladera RUSLE** (distancia D8 al primer
cauce, con piso igual al tamaño de celda).

Cada paso verifica la existencia de su salida para evitar recómputo.


In [ ]:
wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(str(DIR_RASTERS))
wbt.verbose = False

# 7.1 Corrección hidrológica del DEM (breaching de depresiones)
if not DEM_BREACH_FILE.exists():
    wbt.breach_depressions_least_cost(str(DEM_FILE), str(DEM_BREACH_FILE), dist=BREACH_DIST)
    print('DEM hidrológicamente corregido (breach) generado.')
else:
    print('DEM breach ya existe; se omite recómputo.')

# 7.2 Dirección de flujo D8
if not D8_POINTER_FILE.exists():
    wbt.d8_pointer(str(DEM_BREACH_FILE), str(D8_POINTER_FILE))
    print('D8 pointer generado.')
else:
    print('D8 pointer ya existe; se omite recómputo.')

# 7.3 Acumulación de flujo D8 (nº de celdas que drenan a cada celda)
if not FLOW_ACC_FILE.exists():
    wbt.d8_flow_accumulation(str(D8_POINTER_FILE), str(FLOW_ACC_FILE), pntr=True)
    print('Acumulación de flujo generada.')
else:
    print('Acumulación de flujo ya existe; se omite recómputo.')

# 7.4 Distancia de flujo a la salida de la cuenca (variable hidrológica auxiliar,
#     NO es la lambda de RUSLE; ver Sección 11-A).
if not FLOW_LENGTH_FILE.exists():
    wbt.downslope_flowpath_length(d8_pntr=str(D8_POINTER_FILE), output=str(FLOW_LENGTH_FILE))
    print('Longitud de flujo a la salida generada.')
else:
    print('Longitud de flujo a la salida ya existe; se omite recómputo.')

# 7.5 Red de drenaje (cauces) por umbral de acumulación de flujo
if not STREAMS_FILE.exists():
    wbt.extract_streams(str(FLOW_ACC_FILE), str(STREAMS_FILE), threshold=STREAM_THRESHOLD)
    print(f'Red de drenaje generada (umbral {STREAM_THRESHOLD}).')
else:
    print('Red de drenaje ya existe; se omite recómputo.')

# 7.6 Longitud de ladera RUSLE = distancia D8 al primer cauce receptor,
#     con piso físico igual al tamaño de celda (evita ceros en laderas).
if not FLOW_LENGTH_RUSLE_FILE.exists():
    wbt.downslope_distance_to_stream(dem=str(DEM_BREACH_FILE),
                                     streams=str(STREAMS_FILE),
                                     output=str(FLOW_LENGTH_RUSLE_FILE))
    data, perfil, nodata = leer_raster(FLOW_LENGTH_RUSLE_FILE)
    data = np.where((data < CELLSIZE_M) & (~np.isnan(data)), CELLSIZE_M, data)
    guardar_raster(FLOW_LENGTH_RUSLE_FILE, data, perfil, nodata=nodata)
    print(f'Longitud de ladera RUSLE generada y corregida (piso {CELLSIZE_M:.0f} m).')
else:
    print('Longitud de ladera RUSLE ya existe; se omite recómputo.')

print('\nProcesamiento hidrológico completado.')


In [ ]:
# 7.7 Verificación rápida de insumos raster (instantánea del estado actual)
rasters_requeridos = {
    'DEM Copernicus':                DEM_FILE,
    'DEM breach':                    DEM_BREACH_FILE,
    'D8 pointer':                    D8_POINTER_FILE,
    'Acumulación de flujo':          FLOW_ACC_FILE,
    'Longitud de flujo a la salida': FLOW_LENGTH_FILE,
    'Red de drenaje (streams)':      STREAMS_FILE,
    'Longitud de ladera RUSLE':      FLOW_LENGTH_RUSLE_FILE,
    'Pendiente (grados)':            SLOPE_DEG_FILE,
    'Factor LS (Mitasova)':          LS_FILE,
    'CHIRPS climatología mensual':   CHIRPS_MENSUAL_FILE,
}
print('=== VERIFICACIÓN DE INSUMOS RASTER ===')
for nombre, ruta in rasters_requeridos.items():
    print(f'  [{"OK" if ruta.exists() else "PENDIENTE":9s}] {nombre}: {ruta.name}')


## 8. DEM: carga y visualización


In [ ]:
dem, dem_perfil, dem_nodata = leer_raster(DEM_FILE)
pixel_size = dem_perfil['transform'][0]

print('=== INFORMACIÓN DEL DEM ===')
print('CRS         :', dem_perfil['crs'])
print('Resolución  :', pixel_size, 'm')
print('Dimensiones :', dem_perfil['width'], 'x', dem_perfil['height'], 'px')
print('Bandas      :', dem_perfil['count'])
reporte_nodata(dem, 'DEM')


In [ ]:
# Visualización rápida: DEM + hillshade
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

img0 = axes[0].imshow(dem, cmap='terrain')
axes[0].set_title('Modelo Digital de Elevación — Cuenca Amaime', fontsize=12, fontweight='bold')
axes[0].axis('off')
fig.colorbar(img0, ax=axes[0], label='Elevación (m s.n.m.)', fraction=0.046, pad=0.04, shrink=0.85)

ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(dem, vert_exag=1)
axes[1].imshow(hillshade, cmap='gray')
axes[1].set_title('Hillshade (Az=315°, Alt=45°)', fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.tight_layout(rect=[0, 0, 0.98, 1])
plt.show()


## 9. Pendiente del terreno

Se calcula en **grados** a partir del DEM (si el raster aún no existe) y se valida
su rango físico [0°, 90°]. La conversión a **porcentaje** ($\tan\theta\times100$)
se aplica en la extracción a puntos (Sección 11).


In [ ]:
if not SLOPE_DEG_FILE.exists():
    slope_deg = calcular_pendiente_grados(dem, pixel_size)
    guardar_raster(SLOPE_DEG_FILE, slope_deg, dem_perfil)
    print(f'Pendiente en grados calculada y guardada en: {SLOPE_DEG_FILE.name}')
else:
    slope_deg, _, _ = leer_raster(SLOPE_DEG_FILE)
    print(f'Pendiente en grados cargada desde: {SLOPE_DEG_FILE.name}')

reporte_nodata(slope_deg, 'Pendiente (grados)')
verificar_rango_fisico(np.nanmin(slope_deg), np.nanmax(slope_deg), 0, 90, 'Pendiente (grados)')

fig, ax = plt.subplots(figsize=(9, 7))
img = ax.imshow(slope_deg, cmap='terrain')
fig.colorbar(img, ax=ax, label='Pendiente (°)', fraction=0.046, pad=0.04, shrink=0.85)
ax.set_title('Pendiente derivada del DEM', fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout(rect=[0, 0, 0.98, 1])
plt.show()


## 10. Factor topográfico LS — Mitasova & Mitas (2001)

$$LS = \left(\frac{A_s}{22.13}\right)^{m}\left(\frac{\sin\theta}{0.0896}\right)^{n},
\qquad m=0.4,\; n=1.3$$

con $A_s$ el área específica de captación (aproximada como
`flow_accum × cellsize`). Esta formulación incorpora la
**convergencia/divergencia del flujo** y es más adecuada en terrenos complejos
que la clásica de Wischmeier/Smith. Si `LS_Mitasova_2001.tif` ya existe en disco,
se reutiliza. Al depender de un área de captación no acotada, puede producir
valores muy altos en celdas de cauce (ver diagnóstico en la Sección 10‑A).


In [ ]:
flow_acc, _, _ = leer_raster(FLOW_ACC_FILE)

if not LS_FILE.exists():
    LS = calcular_factor_ls_mitasova(flow_acc, slope_deg, CELLSIZE_M, m=LS_M_EXP, n=LS_N_EXP)
    guardar_raster(LS_FILE, LS, dem_perfil)
    print(f'Factor LS calculado y guardado en: {LS_FILE.name}')
else:
    LS, _, _ = leer_raster(LS_FILE)
    print(f'Factor LS cargado desde: {LS_FILE.name}')

reporte_nodata(LS, 'Factor LS')
resumen_ls = resumen_percentiles_completo(LS.flatten(), 'Factor LS')
print(pd.Series(resumen_ls))


In [ ]:
# Umbrales de clasificación del Factor LS — Lu et al. (2020), aplicados en el estudio:
#   muy bajo <1.5 · bajo 1.5–3.0 · moderado 3.0–5.0 · alto 5.0–7.0 · muy alto >7.0
LS_CLASE_BORDES = [0, 1.5, 3.0, 5.0, 7.0, np.inf]
LS_CLASE_ETIQUETAS = ['Muy bajo', 'Bajo', 'Moderado', 'Alto', 'Muy alto']


def clasificar_ls(ls_data):
    """Clasifica LS en 5 categorías con los umbrales fijos de Lu et al. (2020)."""
    ls_data = np.asarray(ls_data, dtype='float64')
    c = np.full(ls_data.shape, np.nan)
    for i, (lo, hi) in enumerate(zip(LS_CLASE_BORDES[:-1], LS_CLASE_BORDES[1:]), start=1):
        c = np.where((ls_data >= lo) & (ls_data < hi), i, c)
    return c


fig, axes = plt.subplots(1, 2, figsize=(17, 7))

img0 = axes[0].imshow(LS, cmap='terrain', vmin=0, vmax=np.nanpercentile(LS, 99))
axes[0].set_title('Factor LS', fontsize=12, fontweight='bold')
axes[0].axis('off')
fig.colorbar(img0, ax=axes[0], label='Factor LS', fraction=0.046, pad=0.04, shrink=0.85)

LS_clase = clasificar_ls(LS)
colores_clase = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027']
etiquetas_clase = ['Muy bajo', 'Bajo', 'Moderado', 'Alto', 'Muy alto']
cmap_clase = plt.matplotlib.colors.ListedColormap(colores_clase)
axes[1].imshow(LS_clase, cmap=cmap_clase, vmin=0.5, vmax=5.5)
axes[1].set_title('Clasificación cualitativa del Factor LS', fontsize=12, fontweight='bold')
axes[1].axis('off')

parches = [plt.matplotlib.patches.Patch(facecolor=c, edgecolor='black', label=l)
           for c, l in zip(colores_clase, etiquetas_clase)]
axes[1].legend(handles=parches, title='Riesgo topográfico', loc='center left',
               bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=9, title_fontsize=10)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()


### 10‑A. Diagnóstico de valores extremos del factor LS

Las formulaciones de LS basadas en área unitaria de contribución (Moore & Burch,
Desmet & Govers, Mitasova & Mitas) son matemáticamente **no acotadas**: como
$A_s$ crece varios órdenes de magnitud en celdas de cauce, LS puede alcanzar
valores de decenas a cientos en esas celdas. Eso **no representa erosión laminar
de ladera**. La práctica estándar es **enmascarar las celdas de cauce** antes de
interpretar LS. Se cuantifica a continuación qué fracción de píxeles supera
LS > 50, 75 y 100 y cuánta coincide con la red de drenaje.


In [ ]:
_ = diagnostico_valores_extremos_ls(
    LS, flow_acc=flow_acc, umbral_red=UMBRAL_RED_DRENAJE, umbrales=LS_UMBRALES_EXTREMOS
)


## 11. Extracción de variables topográficas a los puntos de muestreo

Se construye el `GeoDataFrame` de puntos (CRS proyectado) y se extraen, para cada
punto: **LS** (Mitasova), **pendiente** en grados y en porcentaje,
**acumulación de flujo** (informativa) y **longitud de ladera** (`Longitud_Ladera_m`,
distancia D8 al primer cauce, con piso de un tamaño de celda). Luego se calculan
los **factores L y S clásicos** y el **LS clásico**.


In [ ]:
# 11.1 Extracción unificada de variables topográficas
puntos_gdf = construir_geodataframe(df_puntos, COL_X_PROJ, COL_Y_PROJ, CRS_PROYECTADO)
puntos_gdf = agregar_variables_topograficas_unificada(
    puntos_gdf, LS_FILE, SLOPE_DEG_FILE, FLOW_ACC_FILE, FLOW_LENGTH_RUSLE_FILE
)

print('Extracción unificada completada y validada.')
print('Registros con longitud 0:', (puntos_gdf['Longitud_Ladera_m'] == 0).sum())
display(puntos_gdf[['Longitud_Ladera_m', 'Pendiente_pct', 'FlowAccum']].describe())


In [ ]:
# 11.2 Factores L y S clásicos + LS clásico
puntos_gdf = calcular_factores_ls_clasicos(puntos_gdf)

# Limpieza de variables redundantes heredadas de la tabla de entrada
columnas_a_borrar = ['LS', 'Longitud_Ladera_RUSLE_m']
puntos_gdf = puntos_gdf.drop(columns=[c for c in columnas_a_borrar if c in puntos_gdf.columns])

print('Auditoría — 20 mayores longitudes de ladera:')
_cols_audit = [c for c in ['Longitud_Ladera_m', 'm_param', 'Factor_L', 'Factor_S', 'LS_Clasico', 'R']
               if c in puntos_gdf.columns]
display(puntos_gdf[_cols_audit].sort_values(by='Longitud_Ladera_m', ascending=False).head(20))


In [ ]:
# 11.3 Revisión estadística de las variables topográficas
columnas_revision = ['Longitud_Ladera_m', 'LS_Clasico', 'LS_Mitasova', 'Pendiente_pct']

print('=== ESTADÍSTICAS DESCRIPTIVAS: VARIABLES TOPOGRÁFICAS ===')
stats_topografia = puntos_gdf[columnas_revision].describe().T
stats_topografia['P90'] = puntos_gdf[columnas_revision].quantile(0.9)
stats_topografia['P95'] = puntos_gdf[columnas_revision].quantile(0.95)
display(stats_topografia.round(3))

ls_max = puntos_gdf['LS_Clasico'].max()
print(f'\nDiagnóstico: el factor LS clásico máximo es {ls_max:.2f}.')
if ls_max > 100:
    print('AVISO: se detectan valores de LS muy altos (normal en alta pendiente o convergencia de flujo).')
else:
    print('Los valores de LS están dentro del rango esperado para la zona de estudio.')


### 11‑A. Validación geomorfológica de la longitud de ladera

`wbt.downslope_flowpath_length` calcula la **distancia de flujo D8 hasta la salida
de la cuenca**, no hasta el primer cauce; por eso puede dar decenas de km en la
parte alta de la cuenca — coherente con la geometría del Amaime, pero **no** con
la $\lambda$ de RUSLE. La variable oficial usada aquí (`Longitud_Ladera_m`) es la
**distancia D8 al primer cauce** (umbral 50 celdas), acotada y consistente con el
concepto de longitud de ladera. Además, el factor LS efectivamente usado
(Mitasova & Mitas, 2001) depende del área de captación $A_s$, **no** de una
$\lambda$ explícita, por lo que el resultado de LS no se ve afectado por esta
distinción.


In [ ]:
# Referencias de validación
LADERA_RUSLE_REF_MAX_M = 400.0    # máximo práctico para lambda en RUSLE (USLE Handbook)
FLUJO_SALIDA_REF_MAX_M = 60000.0  # máximo plausible para la distancia de flujo a la salida (60 km)

if 'Longitud_Ladera_m' in puntos_gdf.columns:
    resumen_dfo = resumen_percentiles_completo(puntos_gdf['Longitud_Ladera_m'],
                                               'Longitud de ladera (m)')
    print('Resumen estadístico completo — Longitud de ladera (distancia D8 al primer cauce):')
    print(pd.DataFrame([resumen_dfo]).set_index('variable').T)

    pct_excede_lambda = 100 * (puntos_gdf['Longitud_Ladera_m'] > LADERA_RUSLE_REF_MAX_M).mean()
    print(f'\n[QC geomorfológico] {pct_excede_lambda:.1f}% de los puntos superan el máximo '
          f'práctico de lambda RUSLE ({LADERA_RUSLE_REF_MAX_M:.0f} m).')

    verificar_rango_fisico(
        puntos_gdf['Longitud_Ladera_m'].min(), puntos_gdf['Longitud_Ladera_m'].max(),
        0, LADERA_RUSLE_REF_MAX_M, 'Longitud de ladera (m)'
    )
else:
    print("Aviso: la columna 'Longitud_Ladera_m' no está presente en el GeoDataFrame.")


## 12. Precipitación (CHIRPS) y Factor R

Se extraen las 12 bandas mensuales de la climatología CHIRPS (mm/mes) en
coordenadas geográficas para cada punto y se calculan **P_anual**, **MFI** y
**Factor R** con `calcular_mfi_y_r()`.


In [ ]:
if not CHIRPS_MENSUAL_FILE.exists():
    print(f'ERROR: no se encuentra el archivo CHIRPS: {CHIRPS_MENSUAL_FILE}')
else:
    # 1. Extracción de las 12 bandas mensuales (en coordenadas geográficas)
    for mes in range(1, 13):
        col_name = f'P_{mes:02d}'
        puntos_geo = construir_geodataframe(df_puntos, COL_LON, COL_LAT, CRS_GEOGRAFICO)
        puntos_geo = extraer_valor_raster_puntos(puntos_geo, CHIRPS_MENSUAL_FILE, col_name, banda=mes)
        puntos_gdf[col_name] = puntos_geo[col_name].values

    # 2. MFI y Factor R
    p_anual, mfi, factor_r = calcular_mfi_y_r(puntos_gdf)
    puntos_gdf['P_anual'] = p_anual
    puntos_gdf['MFI'] = mfi
    puntos_gdf['Factor_R'] = factor_r

    # 3. Auditoría de valores
    print('=== AUDITORÍA FACTOR R (MFI) ===')
    print(puntos_gdf[['MFI', 'Factor_R']].describe().T[['min', 'max', 'mean']])
    for col in ['MFI', 'Factor_R']:
        n_nan = puntos_gdf[col].isna().sum()
        n_inf = np.isinf(puntos_gdf[col]).sum()
        print(f'  [{col}] Problemas: NaN={n_nan}, Inf={n_inf}')

    # 4. Visualización
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    puntos_gdf['MFI'].hist(ax=ax[0], color='skyblue', bins=15)
    ax[0].set_title('Distribución MFI')
    puntos_gdf['Factor_R'].hist(ax=ax[1], color='salmon', bins=15)
    ax[1].set_title('Distribución Factor R')
    plt.tight_layout()
    plt.show()


### 12‑A. Análisis mensual de la precipitación y la erosividad (2010–2025)

El Factor R es **anual** por definición (el MFI necesita los 12 meses del año),
pero se puede **descomponer mes a mes**: cada mes aporta $P_i^2/P_{anual}$ al MFI y,
proporcionalmente ($P_i^2/\sum P_i^2$), al Factor R del año. Se extrae de CHIRPS la
**precipitación mensual media de la cuenca** para los 192 meses del periodo y se
analiza su relación con la erosividad.

**Resultado clave:** la lluvia y la erosividad están **fuertemente relacionadas**
(r ≈ 0,96 anual; r ≈ 0,996 entre puntos), pero la erosividad **amplifica** la
variación de la lluvia por su dependencia cuadrática (un CV del ~19 % en la lluvia
anual se traduce en ~42 % en R).


In [ ]:
import matplotlib.dates as mdates

MESES_ES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
SERIE_MENSUAL_CSV = DIR_TABLAS / 'Serie_mensual_P_erosividad_2010_2025.csv'
_Y0, _Y1 = 2010, 2025

# Región = zona de influencia (media de cuenca)
_region_m = ee.Geometry(zona_gdf.to_crs(CRS_GEOGRAFICO).geometry.iloc[0].__geo_interface__)
_chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
           .filterDate(f'{_Y0}-01-01', f'{_Y1 + 1}-01-01')
           .filterBounds(_region_m).select('precipitation'))
_pares = ee.List([[y, m] for y in range(_Y0, _Y1 + 1) for m in range(1, 13)])


def _suma_mes(par):
    par = ee.List(par)
    y, m = ee.Number(par.get(0)), ee.Number(par.get(1))
    img = (_chirps.filter(ee.Filter.calendarRange(y, y, 'year'))
           .filter(ee.Filter.calendarRange(m, m, 'month')).sum())
    val = img.reduceRegion(ee.Reducer.mean(), _region_m, 5566, maxPixels=1e9).get('precipitation')
    return ee.Feature(None, {'anio': y, 'mes': m, 'P_mm': val})


_rows = ee.FeatureCollection(_pares.map(_suma_mes)).getInfo()['features']
serie_mes = pd.DataFrame([r['properties'] for r in _rows]).sort_values(['anio', 'mes']).reset_index(drop=True)
serie_mes['P_mm'] = pd.to_numeric(serie_mes['P_mm'], errors='coerce')
serie_mes['fecha'] = pd.to_datetime(dict(year=serie_mes.anio, month=serie_mes.mes, day=15))

# MFI y R anuales (fórmula establecida) + reparto mensual de la erosividad
_ann = serie_mes.groupby('anio').agg(P_anual=('P_mm', 'sum'),
                                     sumP2=('P_mm', lambda s: (s ** 2).sum())).reset_index()
_ann['MFI'] = _ann.sumP2 / _ann.P_anual
_ann['R'] = np.where(_ann.MFI < 55, 0.7397 * _ann.MFI ** 1.847,
                     95.77 - 6.081 * _ann.MFI + 0.4770 * _ann.MFI ** 2)
serie_mes = serie_mes.merge(_ann, on='anio')
serie_mes['aporte_MFI'] = serie_mes.P_mm ** 2 / serie_mes.P_anual
serie_mes['R_mes'] = serie_mes.R * (serie_mes.P_mm ** 2 / serie_mes.sumP2)   # suma_mes(R_mes) = R_anual
serie_mes.to_csv(SERIE_MENSUAL_CSV, index=False)

clim_mes = serie_mes.groupby('mes').agg(P_mm=('P_mm', 'mean'), P_sd=('P_mm', 'std'),
                                        R_mes=('R_mes', 'mean')).reset_index()
_r_anual = np.corrcoef(_ann.P_anual, _ann.R)[0, 1]
print(f'Meses obtenidos: {len(serie_mes)}   |   corr(P_anual, R) anual = {_r_anual:.3f}')
print(f'CV lluvia anual = {100 * _ann.P_anual.std() / _ann.P_anual.mean():.1f}%   |   '
      f'CV Factor R anual = {100 * _ann.R.std() / _ann.R.mean():.1f}%')
print('\nClimatología mensual media 2010-2025:')
display(clim_mes.assign(mes=[MESES_ES[i - 1] for i in clim_mes.mes]).round(1))


In [ ]:
# FIG 1 — Climatología mensual: precipitación (barras) vs erosividad (línea)
fig, ax1 = plt.subplots(figsize=(11, 5.5))
ax1.bar(clim_mes.mes, clim_mes.P_mm, yerr=clim_mes.P_sd, capsize=3, color='#4575b4', alpha=.8)
ax1.set_xticks(range(1, 13)); ax1.set_xticklabels(MESES_ES)
ax1.set_ylabel('Precipitación media mensual (mm)', color='#4575b4')
ax1.tick_params(axis='y', colors='#4575b4')
ax2 = ax1.twinx()
ax2.plot(clim_mes.mes, clim_mes.R_mes, 'o-', color='#d73027', lw=2.2)
ax2.set_ylabel('Erosividad mensual — aporte al Factor R', color='#d73027')
ax2.tick_params(axis='y', colors='#d73027')
ax1.set_title('Climatología mensual 2010–2025 — la erosividad se concentra en los meses lluviosos',
              fontweight='bold')
ax1.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()

# FIG 2 — Serie mensual completa (192 meses)
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].fill_between(serie_mes.fecha, serie_mes.P_mm, color='#4575b4', alpha=.35)
axes[0].plot(serie_mes.fecha, serie_mes.P_mm, color='#2c5aa0', lw=1)
axes[0].set_ylabel('Precipitación\nmensual (mm)')
axes[0].set_title('Serie mensual 2010–2025 (CHIRPS, media de cuenca)', fontweight='bold')
axes[1].fill_between(serie_mes.fecha, serie_mes.R_mes, color='#d73027', alpha=.35)
axes[1].plot(serie_mes.fecha, serie_mes.R_mes, color='#a01e1e', lw=1)
axes[1].set_ylabel('Erosividad mensual\n(aporte a R)'); axes[1].set_xlabel('Año')
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for ax in axes:
    ax.grid(alpha=.25)
plt.tight_layout(); plt.show()


In [ ]:
# FIG 3 — Relación precipitación mensual -> erosividad mensual (192 meses)
fig, ax = plt.subplots(figsize=(8.5, 6))
_sc = ax.scatter(serie_mes.P_mm, serie_mes.R_mes, c=serie_mes.mes, cmap='twilight',
                 s=32, edgecolor='0.3', lw=.3)
_xx = np.linspace(0, serie_mes.P_mm.max(), 100)
ax.plot(_xx, np.polyval(np.polyfit(serie_mes.P_mm, serie_mes.R_mes, 2), _xx), 'k--', lw=1.5)
_rp = np.corrcoef(serie_mes.P_mm, serie_mes.R_mes)[0, 1]
_rp2 = np.corrcoef(serie_mes.P_mm ** 2, serie_mes.R_mes)[0, 1]
ax.set_xlabel('Precipitación mensual (mm)'); ax.set_ylabel('Erosividad mensual (aporte a R)')
ax.set_title(f'La erosividad crece con el cuadrado de la lluvia mensual\n'
             f'r(P, erosiv.) = {_rp:.2f}   ·   r(P², erosiv.) = {_rp2:.2f}   (192 meses)', fontweight='bold')
_cb = fig.colorbar(_sc, ax=ax, ticks=range(1, 13)); _cb.ax.set_yticklabels(MESES_ES); _cb.set_label('Mes')
ax.grid(alpha=.25); plt.tight_layout(); plt.show()

# FIG 4 — Anual: series normalizadas + dispersión P_anual vs R (enlace con Obj. V)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(_ann.anio, _ann.P_anual / _ann.P_anual.mean(), 'o-', color='#4575b4', lw=2,
           label='Precipitación anual (÷ media)')
ax[0].plot(_ann.anio, _ann.R / _ann.R.mean(), 's-', color='#d73027', lw=2,
           label='Factor R anual (÷ media)')
ax[0].axhline(1, color='0.6', lw=.8)
ax[0].set_title('Series normalizadas: R sigue a la lluvia con oscilaciones más amplias', fontweight='bold')
ax[0].set_xticks(_ann.anio); ax[0].set_xticklabels(_ann.anio, rotation=45)
ax[0].set_ylabel('valor / media del periodo'); ax[0].legend(); ax[0].grid(alpha=.25)
ax[1].scatter(_ann.P_anual, _ann.R, s=45, color='#7b3294', zorder=3)
_xx = np.linspace(_ann.P_anual.min(), _ann.P_anual.max(), 100)
ax[1].plot(_xx, np.polyval(np.polyfit(_ann.P_anual, _ann.R, 2), _xx), 'k--', lw=1.4)
ax[1].set_xlabel('Precipitación anual (mm)'); ax[1].set_ylabel('Factor R anual')
ax[1].set_title(f'Precipitación anual vs Factor R\nr = {_r_anual:.2f}  (n = 16 años)', fontweight='bold')
ax[1].grid(alpha=.25)
plt.tight_layout(); plt.show()


### 12‑B. Estadística descriptiva y correlación de la precipitación mensual (2010–2025) con las pérdidas de suelo RUSLE (A)

Se extrae de CHIRPS la **climatología mensual del periodo 2010–2025** (16 años) en
los **117 puntos** (`Pm_Ene … Pm_Dic`, media mensual en mm; `Pm_anual`) y se relaciona
con la **pérdida de suelo `RUSLE (A)`** de la hoja `Datos` de la base de la tesis
(t·ha⁻¹·año⁻¹), es decir el resultado del modelo A = R·K·LS·C·P ya reportado.

1. **Estadística descriptiva** (tendencia central y dispersión) de cada variable:
   media, mediana, moda, desviación estándar, coeficiente de variación, cuartiles,
   rango, IQR, asimetría y curtosis.
2. **Prueba de normalidad — Shapiro‑Wilk** sobre cada variable. Si todas son
   normales (p ≥ 0,05) la correlación se hace con **Pearson**; si alguna no lo es,
   con **Spearman**.
3. **Matriz de correlación** (Pm mensual, Pm anual, RUSLE (A)) con ese método,
   incluyendo p‑valores.
4. **Cada gráfico se guarda como archivo independiente** en `Salida/figuras/`:
   `Estadistica_descriptiva_precip_A.png`, `Normalidad_QQ_Pm_anual.png`,
   `Normalidad_QQ_A_RUSLE.png`, `Matriz_correlacion_precip_A.png`,
   `Correlacion_P_mensual_vs_A.png`, `Dispersion_P_anual_vs_A.png`.

> Se correlaciona con **RUSLE (A)** (pérdida de suelo), **no con el Factor R**. El
> MFI tampoco se incluye. El interés es la relación **precipitación → pérdida de suelo**.


In [ ]:
from scipy import stats

# Descargar la climatología mensual CHIRPS 2010-2025 y extraerla en los puntos
if FORZAR_DESCARGA_EE or not CHIRPS_MENSUAL_CORR_FILE.exists():
    descargar_chirps_mensual(AOI, CHIRPS_MENSUAL_CORR_FILE,
                             anio_ini=CORR_ANIO_INI, anio_fin=CORR_ANIO_FIN, crs=CRS_GEOGRAFICO)
else:
    print(f'Climatología {CORR_ANIO_INI}-{CORR_ANIO_FIN} ya presente: {CHIRPS_MENSUAL_CORR_FILE.name}')

_mesg = ['Pm_Ene', 'Pm_Feb', 'Pm_Mar', 'Pm_Abr', 'Pm_May', 'Pm_Jun',
         'Pm_Jul', 'Pm_Ago', 'Pm_Sep', 'Pm_Oct', 'Pm_Nov', 'Pm_Dic']
_pg = construir_geodataframe(df_puntos, COL_LON, COL_LAT, CRS_GEOGRAFICO)
for _m in range(1, 13):
    _pg = extraer_valor_raster_puntos(_pg, CHIRPS_MENSUAL_CORR_FILE, _mesg[_m - 1], banda=_m)
_pm = _pg[_mesg].copy()
_pm['Pm_anual'] = _pm[_mesg].sum(axis=1)

# Pérdida de suelo RUSLE (A) del estudio (hoja 'Datos' de la base de la tesis),
# unida por ID_UNAL. Es el resultado del modelo A = R·K·LS·C·P ya reportado.
_sa = pd.read_excel(ARCHIVO_SUELOS, sheet_name='Datos')
_amap = dict(zip(_sa['ID_UNAL'], pd.to_numeric(_sa['RUSLE (A)'], errors='coerce')))
_pm['A_RUSLE'] = df_puntos['ID_UNAL'].map(_amap).values
print(f"RUSLE (A) del estudio unida por ID_UNAL: {_pm['A_RUSLE'].notna().sum()}/117 puntos")

# guardar la precipitación mensual 2010-2025 + RUSLE (A) por punto
_pm_out = pd.concat([df_puntos[[c for c in ['ID_UNAL', 'ID_CIAT', 'LONG', 'LAT',
                                            COL_X_PROJ, COL_Y_PROJ, 'MSNM'] if c in df_puntos.columns]],
                     _pm], axis=1)
_pm_out.to_csv(DIR_TABLAS / 'Precipitacion_mensual_2010_2025_por_punto.csv', index=False)
print(f'Precipitación mensual media {CORR_ANIO_INI}-{CORR_ANIO_FIN} extraída en {len(_pm)} puntos.')

_cols = _mesg + ['Pm_anual', 'A_RUSLE']

# --- 1. Estadística descriptiva (tendencia central y dispersión) ---
def _descriptivos(_s):
    _s = _s.dropna().astype(float)
    _md = _s.round(3).mode()
    return pd.Series({
        'n': int(_s.size),
        'media': _s.mean(), 'mediana': _s.median(),
        'moda': float(_md.iloc[0]) if not _md.empty else np.nan,
        'desv_est': _s.std(ddof=1),
        'CV_%': 100 * _s.std(ddof=1) / _s.mean() if _s.mean() else np.nan,
        'min': _s.min(), 'Q1': _s.quantile(.25), 'Q3': _s.quantile(.75), 'max': _s.max(),
        'rango': _s.max() - _s.min(), 'IQR': _s.quantile(.75) - _s.quantile(.25),
        'asimetria': _s.skew(), 'curtosis': _s.kurt(),
    })

desc = _pm[_cols].apply(_descriptivos).T
desc.to_csv(DIR_TABLAS / 'Estadistica_descriptiva_precip_A.csv')
print('=== Estadística descriptiva — tendencia central y dispersión (n = 117) ===')
display(desc.round(3))

# --- 2. Prueba de normalidad (Shapiro-Wilk) ---
_norm = []
for _v in _cols:
    _w, _p = stats.shapiro(_pm[_v].dropna())
    _norm.append({'variable': _v, 'Shapiro_W': round(_w, 4),
                  'p_value': _p, 'normal_(p>=0.05)': _p >= 0.05})
_norm = pd.DataFrame(_norm)
_norm.to_csv(DIR_TABLAS / 'Normalidad_Shapiro_precip_A.csv', index=False)
print('\n=== Prueba de normalidad — Shapiro-Wilk (n = 117) ===')
display(_norm)

_todas_normales = bool(_norm['normal_(p>=0.05)'].all())
_metodo = 'pearson' if _todas_normales else 'spearman'
_no_norm = _norm.loc[~_norm['normal_(p>=0.05)'], 'variable'].tolist()
print(f"\nVariables NO normales: {_no_norm if _no_norm else 'ninguna'}")
print(f"-> Método de correlación: {_metodo.upper()} "
      f"({'todas normales' if _todas_normales else 'hay variables no normales'})")

# --- 3. Matriz de correlación P mensual / P anual vs RUSLE (A) ---
mat_corr = _pm[_cols].corr(method=_metodo)
mat_corr.to_csv(DIR_TABLAS / 'Matriz_correlacion_precip_A.csv')
_cA = mat_corr['A_RUSLE'].drop('A_RUSLE')
_pcol = 'p_spearman' if _metodo == 'spearman' else 'p_pearson'
_pvals = {c: (stats.spearmanr(_pm[c], _pm['A_RUSLE'])[1] if _metodo == 'spearman'
              else stats.pearsonr(_pm[c], _pm['A_RUSLE'])[1]) for c in _cols[:-1]}
tabla_corr_A = pd.DataFrame({'variable': list(_cA.index),
                             f'r_{_metodo}': _cA.values,
                             _pcol: [_pvals[c] for c in _cA.index]}).round(4)
tabla_corr_A.to_csv(DIR_TABLAS / 'Correlacion_precip_vs_A.csv', index=False)
print(f'\nCorrelación ({_metodo}) precipitación ↔ RUSLE (A):')
display(tabla_corr_A)

# --- 4. Figuras INDEPENDIENTES (una por archivo) ---
_MES3 = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
_ETI = _MES3 + ['P anual', 'RUSLE (A)']

# 4a) Estadística descriptiva como tabla-figura
_dv = desc[['media', 'mediana', 'moda', 'desv_est', 'CV_%', 'min', 'max',
            'asimetria', 'curtosis']].copy()
fig, _axt = plt.subplots(figsize=(11, 4.6)); _axt.axis('off')
_tab = _axt.table(cellText=[[f'{x:,.2f}' for x in _dv.loc[v]] for v in _dv.index],
                  rowLabels=_ETI, colLabels=list(_dv.columns),
                  cellLoc='center', rowLoc='center', loc='center')
_tab.auto_set_font_size(False); _tab.set_fontsize(8); _tab.scale(1, 1.35)
for _j in range(len(_dv.columns)):
    _tab[0, _j].set_text_props(weight='bold'); _tab[0, _j].set_facecolor('#dfe7f2')
_axt.set_title('Estadística descriptiva — precipitación mensual 2010–2025 y RUSLE (A)',
               fontweight='bold', pad=14)
plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Estadistica_descriptiva_precip_A.png', dpi=150, bbox_inches='tight')
plt.show()

# 4b) Q-Q de normalidad — un archivo por variable
for _v, _fn in [('Pm_anual', 'Normalidad_QQ_Pm_anual'), ('A_RUSLE', 'Normalidad_QQ_A_RUSLE')]:
    fig, _ax = plt.subplots(figsize=(6.5, 5.5))
    stats.probplot(_pm[_v], dist='norm', plot=_ax)
    _pv = float(_norm.loc[_norm.variable == _v, 'p_value'].iloc[0])
    _ax.get_lines()[0].set(marker='o', markersize=4, alpha=.7)
    _ax.set_title(f'Normalidad (Q-Q) — {_v}\nShapiro-Wilk  p = {_pv:.3g}  '
                  f'({"normal" if _pv >= 0.05 else "NO normal"})', fontweight='bold')
    _ax.grid(alpha=.3); plt.tight_layout()
    fig.savefig(DIR_FIGURAS / f'{_fn}.png', dpi=150, bbox_inches='tight')
    plt.show()

# 4c) Matriz de correlación (heatmap)
fig, _axm = plt.subplots(figsize=(9.5, 8.5))
_im = _axm.imshow(mat_corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
_axm.set_xticks(range(len(_cols))); _axm.set_xticklabels(_ETI, rotation=45, ha='right')
_axm.set_yticks(range(len(_cols))); _axm.set_yticklabels(_ETI)
for _i in range(len(_cols)):
    for _j in range(len(_cols)):
        _axm.text(_j, _i, f'{mat_corr.values[_i, _j]:.2f}', ha='center', va='center',
                  fontsize=7, color='white' if abs(mat_corr.values[_i, _j]) > .6 else 'black')
_axm.set_title(f'Matriz de correlación ({_metodo}) — precipitación mensual 2010–2025 y RUSLE (A)',
               fontweight='bold')
fig.colorbar(_im, ax=_axm, fraction=0.046, pad=0.04, label='coeficiente de correlación')
plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Matriz_correlacion_precip_A.png', dpi=150, bbox_inches='tight')
plt.show()

# 4d) Barras: r de cada mes con RUSLE (A)
fig, _axb = plt.subplots(figsize=(10, 5))
_axb.bar(range(12), _cA[_mesg].values, color='#4575b4')
_axb.axhline(_cA['Pm_anual'], color='#d73027', ls='--', lw=1.8,
             label=f'P anual (r = {_cA["Pm_anual"]:.3f})')
_ytop = max(float(_cA[_mesg].max()), float(_cA['Pm_anual']), 0.05) * 1.45
for _bi, _bv in enumerate(_cA[_mesg].values):
    _axb.text(_bi, _bv + _ytop * .02, f'{_bv:.2f}', ha='center', fontsize=8)
_axb.set_ylim(0, _ytop)
_axb.set_xticks(range(12)); _axb.set_xticklabels(_MES3)
_axb.set_ylabel(f'r ({_metodo}) con RUSLE (A)')
_axb.set_title('Correlación de la precipitación mensual (2010–2025) con la pérdida de suelo RUSLE (A)',
               fontweight='bold')
_axb.legend(); _axb.grid(axis='y', alpha=.3); plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Correlacion_P_mensual_vs_A.png', dpi=150, bbox_inches='tight')
plt.show()

# 4e) Dispersión P anual vs RUSLE (A) — escala log en A por el fuerte sesgo
fig, _axs = plt.subplots(figsize=(7, 6))
_axs.scatter(_pm['Pm_anual'], _pm['A_RUSLE'].clip(lower=1e-2), s=28, alpha=.7, color='#7b3294')
_axs.set_yscale('log')
_axs.set_xlabel('Precipitación anual media 2010–2025 (mm)')
_axs.set_ylabel('RUSLE (A)  [t·ha⁻¹·año⁻¹]  (escala log)')
_axs.set_title(f'Precipitación anual vs RUSLE (A)\n'
               f'r ({_metodo}) = {_cA["Pm_anual"]:.3f}  (n = 117 puntos)', fontweight='bold')
_axs.grid(alpha=.3, which='both'); plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Dispersion_P_anual_vs_A.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFiguras (Salida/figuras/):')
for _f in ['Estadistica_descriptiva_precip_A', 'Normalidad_QQ_Pm_anual', 'Normalidad_QQ_A_RUSLE',
           'Matriz_correlacion_precip_A', 'Correlacion_P_mensual_vs_A', 'Dispersion_P_anual_vs_A']:
    print(f'  {_f}.png')
print('Tablas (Salida/tablas/): Estadistica_descriptiva_precip_A.csv · Normalidad_Shapiro_precip_A.csv · '
      'Matriz_correlacion_precip_A.csv · Correlacion_precip_vs_A.csv · '
      'Precipitacion_mensual_2010_2025_por_punto.csv')


### 12‑C. Normalidad y correlación de los factores de RUSLE con la pérdida de suelo A

El objetivo es **`RUSLE (A)`**. Aquí se analiza qué factor de la ecuación
`A = R · K · LS · C · P` explica la variación espacial de la pérdida de suelo entre
los 117 puntos, tomando los valores **de la hoja `Datos` de la base de la tesis**.

1. **Estadística descriptiva** de `R, K, LS, C, P, A`.
2. **Prueba de normalidad — Shapiro‑Wilk** sobre **cada** variable. Si todas son
   normales (p ≥ 0,05) → **Pearson**; si alguna no lo es → **Spearman**.
3. **Matriz de correlación** de los seis con ese método (+ p‑valores) y **correlación
   de cada factor con A**.
4. **Descomposición de la varianza de `ln A = ln R + ln K + ln LS + ln C + ln P`**:
   cuánto aporta cada factor a la varianza total de la pérdida de suelo (suma = 100 %).
5. Cada gráfico en su propio archivo en `Salida/figuras/`.


In [ ]:
from scipy import stats

# --- Factores OFICIALES del estudio (hoja 'Datos'), unidos por ID_UNAL ---
_sae = pd.read_excel(ARCHIVO_SUELOS, sheet_name='Datos')
_kc = [c for c in _sae.columns if 'Factor K calculado' in c][0]
_cc = [c for c in _sae.columns if c.startswith('Factor de Cultivos')][0]
_lc = [c for c in _sae.columns if c.strip() == 'LS Diego'][0]
_rc = [c for c in _sae.columns if c.strip() == 'Factor R'][0]
_pc = [c for c in _sae.columns if c.strip().upper() == 'FACTOR P'][0]
_ac = [c for c in _sae.columns if c.strip() == 'RUSLE (A)'][0]
_mapa = {'R': _rc, 'K': _kc, 'LS': _lc, 'C': _cc, 'P': _pc, 'A': _ac}

_fac = pd.DataFrame({'ID_UNAL': _sae['ID_UNAL']})
for _k, _col in _mapa.items():
    _fac[_k] = pd.to_numeric(_sae[_col], errors='coerce')
_fac = df_puntos[['ID_UNAL']].merge(_fac, on='ID_UNAL', how='left')
_vars = ['R', 'K', 'LS', 'C', 'P', 'A']
_fac = _fac.dropna(subset=_vars).reset_index(drop=True)
print(f'Factores del estudio unidos por ID_UNAL: {len(_fac)}/117 puntos completos')

# --- 1. Estadística descriptiva ---
def _desc_f(_s):
    _s = _s.astype(float)
    return pd.Series({'n': int(_s.size), 'media': _s.mean(), 'mediana': _s.median(),
                      'desv_est': _s.std(ddof=1),
                      'CV_%': 100 * _s.std(ddof=1) / _s.mean() if _s.mean() else np.nan,
                      'min': _s.min(), 'max': _s.max(), 'max/min': _s.max() / _s.min() if _s.min() else np.nan,
                      'asimetria': _s.skew(), 'curtosis': _s.kurt()})

desc_fac = _fac[_vars].apply(_desc_f).T
desc_fac.to_csv(DIR_TABLAS / 'Estadistica_descriptiva_factores_RUSLE.csv')
print('\n=== Estadística descriptiva de los factores ===')
display(desc_fac.round(4))

# --- 2. Normalidad (Shapiro-Wilk) por variable ---
_nf = []
for _v in _vars:
    _w, _p = stats.shapiro(_fac[_v])
    _nf.append({'variable': _v, 'Shapiro_W': round(_w, 4), 'p_value': _p,
                'normal_(p>=0.05)': _p >= 0.05})
_nf = pd.DataFrame(_nf)
_nf.to_csv(DIR_TABLAS / 'Normalidad_Shapiro_factores_RUSLE.csv', index=False)
print('\n=== Prueba de normalidad — Shapiro-Wilk ===')
display(_nf)

_tn = bool(_nf['normal_(p>=0.05)'].all())
_met = 'pearson' if _tn else 'spearman'
_nn = _nf.loc[~_nf['normal_(p>=0.05)'], 'variable'].tolist()
print(f"\nVariables NO normales: {_nn if _nn else 'ninguna'}")
print(f"-> Método de correlación: {_met.upper()} "
      f"({'todas normales' if _tn else 'hay variables no normales'})")

# --- 3. Matriz de correlación y correlación de cada factor con A ---
mat_fac = _fac[_vars].corr(method=_met)
mat_fac.to_csv(DIR_TABLAS / 'Matriz_correlacion_factores_RUSLE.csv')
_cf = mat_fac['A'].drop('A')
_pf = {v: (stats.spearmanr(_fac[v], _fac['A'])[1] if _met == 'spearman'
           else stats.pearsonr(_fac[v], _fac['A'])[1]) for v in _vars[:-1]}
tabla_fac_A = pd.DataFrame({'factor': list(_cf.index), f'r_{_met}': _cf.values,
                            f'p_{_met}': [_pf[v] for v in _cf.index]}).round(4)
tabla_fac_A = tabla_fac_A.sort_values(f'r_{_met}', key=lambda s: s.abs(), ascending=False)
tabla_fac_A.to_csv(DIR_TABLAS / 'Correlacion_factores_vs_A.csv', index=False)
print(f'\n=== Correlación ({_met}) de cada factor con RUSLE (A) ===')
display(tabla_fac_A)

# --- 4. Descomposición de la varianza de ln A ---
_pos = _fac[(_fac[_vars] > 0).all(axis=1)]
_L = np.log(_pos[['R', 'K', 'LS', 'C', 'P']]); _LA = np.log(_pos['A'])
_vLA = _LA.var(ddof=1)
descomp = pd.DataFrame({
    'factor': ['R', 'K', 'LS', 'C', 'P'],
    'var_ln_factor': [_L[c].var(ddof=1) for c in ['R', 'K', 'LS', 'C', 'P']],
    'contrib_var_lnA_%': [100 * np.cov(_L[c], _LA, ddof=1)[0, 1] / _vLA for c in ['R', 'K', 'LS', 'C', 'P']],
}).round(3)
descomp.to_csv(DIR_TABLAS / 'Descomposicion_varianza_lnA.csv', index=False)
print(f'\n=== Descomposición de la varianza de ln(A)  (n = {len(_pos)} puntos con todo > 0) ===')
display(descomp)

# --- 5. Figuras INDEPENDIENTES ---
# 5a) Q-Q de normalidad, un archivo por variable
_ufac = {'R': 'MJ·mm·ha⁻¹·h⁻¹·año⁻¹', 'K': 't·ha·h·ha⁻¹·MJ⁻¹·mm⁻¹', 'LS': '(adim.)',
         'C': '(adim.)', 'P': '(adim.)', 'A': 't·ha⁻¹·año⁻¹'}
for _v in _vars:
    fig, _ax = plt.subplots(figsize=(6.5, 5.5))
    stats.probplot(_fac[_v], dist='norm', plot=_ax)
    _pv = float(_nf.loc[_nf.variable == _v, 'p_value'].iloc[0])
    _ax.get_lines()[0].set(marker='o', markersize=4, alpha=.7)
    _ax.set_title(f'Normalidad (Q-Q) — factor {_v}  {_ufac[_v]}\n'
                  f'Shapiro-Wilk  p = {_pv:.3g}  ({"normal" if _pv >= 0.05 else "NO normal"})',
                  fontweight='bold')
    _ax.grid(alpha=.3); plt.tight_layout()
    fig.savefig(DIR_FIGURAS / f'Normalidad_QQ_fac_{_v}.png', dpi=150, bbox_inches='tight')
    plt.show()

# 5b) Matriz de correlación (heatmap)
fig, _axm = plt.subplots(figsize=(7.2, 6))
_im = _axm.imshow(mat_fac.values, cmap='RdBu_r', vmin=-1, vmax=1)
_axm.set_xticks(range(6)); _axm.set_xticklabels(_vars)
_axm.set_yticks(range(6)); _axm.set_yticklabels(_vars)
for _i in range(6):
    for _j in range(6):
        _axm.text(_j, _i, f'{mat_fac.values[_i, _j]:.2f}', ha='center', va='center',
                  fontsize=9, color='white' if abs(mat_fac.values[_i, _j]) > .6 else 'black')
_axm.set_title(f'Matriz de correlación ({_met}) — factores de RUSLE y A', fontweight='bold')
fig.colorbar(_im, ax=_axm, fraction=0.046, pad=0.04, label='coeficiente de correlación')
plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Matriz_correlacion_factores_RUSLE.png', dpi=150, bbox_inches='tight')
plt.show()

# 5c) Barras: correlación de cada factor con A
fig, _axb = plt.subplots(figsize=(8, 5))
_o = tabla_fac_A.set_index('factor')[f'r_{_met}']
_col = ['#1a9850' if _p < 0.05 else '#bbbbbb' for _p in tabla_fac_A[f'p_{_met}']]
_axb.bar(_o.index, _o.values, color=_col)
for _i, (_f, _v) in enumerate(_o.items()):
    _pp = float(tabla_fac_A.loc[tabla_fac_A.factor == _f, f'p_{_met}'].iloc[0])
    _axb.text(_i, _v + .02, f'{_v:.2f}\n{"p<0.05" if _pp < 0.05 else "n.s."}',
              ha='center', va='bottom', fontsize=8)
_axb.set_ylim(0, max(0.05, float(_o.max())) * 1.35)
_axb.set_ylabel(f'r ({_met}) con RUSLE (A)')
_axb.set_title('Correlación de cada factor de RUSLE con la pérdida de suelo A\n'
               '(verde = significativa p < 0,05; gris = no significativa)', fontweight='bold')
_axb.grid(axis='y', alpha=.3); plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Correlacion_factores_vs_A.png', dpi=150, bbox_inches='tight')
plt.show()

# 5d) Descomposición de la varianza de ln A
fig, _axd = plt.subplots(figsize=(8, 5))
_dd = descomp.sort_values('contrib_var_lnA_%', ascending=False)
_axd.bar(_dd['factor'], _dd['contrib_var_lnA_%'], color='#4575b4')
for _i, _v in enumerate(_dd['contrib_var_lnA_%'].values):
    _axd.text(_i, _v + .8, f'{_v:.1f}%', ha='center', fontsize=9)
_axd.set_ylabel('Contribución a la varianza de ln(A)  [%]')
_axd.set_title('¿Qué factor explica la variación espacial de la pérdida de suelo?\n'
               'Descomposición de var(ln A) = var(ln R + ln K + ln LS + ln C + ln P)',
               fontweight='bold')
_axd.grid(axis='y', alpha=.3); plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Descomposicion_varianza_lnA.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFiguras (Salida/figuras/): Normalidad_QQ_fac_R/K/LS/C/P/A.png · '
      'Matriz_correlacion_factores_RUSLE.png · Correlacion_factores_vs_A.png · '
      'Descomposicion_varianza_lnA.png')
print('Tablas (Salida/tablas/): Estadistica_descriptiva_factores_RUSLE.csv · '
      'Normalidad_Shapiro_factores_RUSLE.csv · Matriz_correlacion_factores_RUSLE.csv · '
      'Correlacion_factores_vs_A.csv · Descomposicion_varianza_lnA.csv')


### 12‑D. Factor C mensual desde Sentinel‑2 (climatología, metodología análoga a la CVC)

La CVC estimó la cobertura por **fotointerpretación CORINE 1:25.000 sobre Sentinel‑2
(2018–2020)** y asignó el **Factor C por clase** (columna `Factor de Cultivos ©`, que
aquí no se modifica). Esta sección **automatiza el mismo insumo** para obtener un
**C mensual** que se pueda cruzar con la precipitación mensual:

- **`COPERNICUS/S2_SR_HARMONIZED`** — reflectancia de superficie (calibración
  radiométrica incluida).
- **Máscara de nubes**: `QA60` (bits 10 = nube, 11 = cirro) + `SCL` (3, 8, 9, 10, 11 =
  sombra / nube / cirro / nieve).
- **Compuesto mensual climatológico**: mediana del NDVI de **todas** las escenas de
  ese mes calendario en el archivo (2017–2025), igual que la climatología de CHIRPS.
- Se calculan **NDVI y EVI** mensuales (`EVI = 2.5·(NIR−RED)/(NIR+6·RED−7.5·BLUE+1)`).
- **NDVI → C**: `C = clip((1 − NDVI) / 2, 0, 1)` — **Durigon et al. (2014)**, trópico.
- `C_anual_pond_EI` = C mensual ponderado por la erosividad mensual (`Pm_i²`).

> Es una **actualización**, no reproduce el C por clase de la CVC. Se escribe en la
> hoja **`Cobertura_C_mensual`** de `Cuenca Amaime Tesis Dayana.xlsx` (sin tocar las
> demás hojas) y en `Salida/tablas/Cobertura_C_mensual.csv`.


In [ ]:
MESES_C = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
COBERTURA_C_CSV = DIR_TABLAS / 'Cobertura_C_mensual.csv'

_fc_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([float(r[COL_LON]), float(r[COL_LAT])]), {'ID_UNAL': r['ID_UNAL']})
    for _, r in df_puntos.iterrows()])
_bbox = _fc_pts.geometry().bounds().buffer(3000)


def _mask_s2_idx(img):
    qa = img.select('QA60')
    _clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    scl = img.select('SCL')
    _sclok = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    _sr = img.select(['B2', 'B4', 'B8']).multiply(1e-4)
    _nir, _red, _blue = _sr.select('B8'), _sr.select('B4'), _sr.select('B2')
    _ndvi = _nir.subtract(_red).divide(_nir.add(_red)).rename('NDVI')
    _evi = (_nir.subtract(_red).multiply(2.5)
            .divide(_nir.add(_red.multiply(6)).subtract(_blue.multiply(7.5)).add(1))
            .rename('EVI'))
    return ee.Image.cat([_ndvi, _evi]).updateMask(_clear).updateMask(_sclok)


if FORZAR_DESCARGA_EE or not COBERTURA_C_CSV.exists():
    _s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate('2017-01-01', '2026-01-01').filterBounds(_bbox)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))
    print(f'Escenas Sentinel-2 SR en la zona (2017-2025): {_s2.size().getInfo()}')
    _cob = {r['ID_UNAL']: {'ID_UNAL': r['ID_UNAL'], 'LONG': r[COL_LON], 'LAT': r[COL_LAT]}
            for _, r in df_puntos.iterrows()}
    for _m in range(1, 13):
        _mc = _s2.filter(ee.Filter.calendarRange(_m, _m, 'month')).map(_mask_s2_idx)
        _sf = _mc.median().reduceRegions(
            collection=_fc_pts, reducer=ee.Reducer.mean(), scale=10, tileScale=4
        ).getInfo()['features']
        for _f in _sf:
            _pp = _f['properties']
            _cob[_pp['ID_UNAL']][f'NDVI_{MESES_C[_m - 1]}'] = _pp.get('NDVI', np.nan)
            _cob[_pp['ID_UNAL']][f'EVI_{MESES_C[_m - 1]}'] = _pp.get('EVI', np.nan)
        print(f'  {MESES_C[_m - 1]}: {_mc.size().getInfo():4d} escenas -> NDVI y EVI muestreados')
    cobertura_c = pd.DataFrame(list(_cob.values()))
    for _x in MESES_C:
        cobertura_c[f'C_{_x}'] = ((1.0 - cobertura_c[f'NDVI_{_x}']) / 2.0).clip(0, 1)  # Durigon 2014

    # C por clase de la CVC (fijo) + cobertura descrita
    _sue = pd.read_excel(ARCHIVO_SUELOS, sheet_name='Datos')
    _ccc = [c for c in _sue.columns if c.startswith('Factor de Cultivos')][0]
    cobertura_c = cobertura_c.merge(
        _sue[['ID_UNAL', _ccc, 'DESC_COBERT_CAM']].rename(
            columns={_ccc: 'C_CVC_clase', 'DESC_COBERT_CAM': 'Cobertura_CVC'}),
        on='ID_UNAL', how='left')

    _ccols = [f'C_{_x}' for _x in MESES_C]
    cobertura_c['C_anual_media'] = cobertura_c[_ccols].mean(axis=1)
    if (DIR_TABLAS / 'Precipitacion_mensual_2010_2025_por_punto.csv').exists():
        _pmc = pd.read_csv(DIR_TABLAS / 'Precipitacion_mensual_2010_2025_por_punto.csv')
        _pcc = [f'Pm_{_x}' for _x in MESES_C]
        _W = _pmc.set_index('ID_UNAL').reindex(cobertura_c['ID_UNAL'])[_pcc].values ** 2
        cobertura_c['C_anual_pond_EI'] = (cobertura_c[_ccols].values * _W).sum(axis=1) / _W.sum(axis=1)

    _ord = (['ID_UNAL', 'LONG', 'LAT', 'Cobertura_CVC', 'C_CVC_clase']
            + [f'NDVI_{_x}' for _x in MESES_C] + [f'EVI_{_x}' for _x in MESES_C] + _ccols
            + ['C_anual_media'] + (['C_anual_pond_EI'] if 'C_anual_pond_EI' in cobertura_c else []))
    cobertura_c = cobertura_c[_ord].round(4)
    cobertura_c.to_csv(COBERTURA_C_CSV, index=False)
    try:
        with pd.ExcelWriter(ARCHIVO_SUELOS, mode='a', engine='openpyxl',
                            if_sheet_exists='replace') as _w:
            cobertura_c.to_excel(_w, sheet_name='Cobertura_C_mensual', index=False)
        print("Hoja 'Cobertura_C_mensual' escrita en la base de la tesis.")
    except PermissionError:
        print('!! Excel abierto: ciérralo y re-ejecuta para inyectar la hoja (CSV ya guardado).')
else:
    cobertura_c = pd.read_csv(COBERTURA_C_CSV)
    print(f'Cobertura C mensual ya presente: {COBERTURA_C_CSV.name}')

_ndc = [f'NDVI_{_x}' for _x in MESES_C]
_evc = [f'EVI_{_x}' for _x in MESES_C]
_cmc = [f'C_{_x}' for _x in MESES_C]
print('\nFactor C mensual (Sentinel-2, Durigon 2014) — resumen:')
display(cobertura_c[['ID_UNAL', 'Cobertura_CVC', 'C_CVC_clase', 'C_anual_media']
                    + (['C_anual_pond_EI'] if 'C_anual_pond_EI' in cobertura_c.columns else [])].head(10))

# --- figuras ---
_ndm = cobertura_c[_ndc].mean().values
_evm = cobertura_c[_evc].mean().values
_cm = cobertura_c[_cmc].mean().values
fig, _ax = plt.subplots(figsize=(9, 4.8))
_ax.plot(MESES_C, _ndm, 'o-', color='#1a9850', label='NDVI medio (117 pts)')
_ax.plot(MESES_C, _evm, '^-', color='#2c7fb8', label='EVI medio (117 pts)')
_ax.set_ylabel('NDVI / EVI'); _ax.set_ylim(0, 1); _ax.grid(alpha=.3)
_ax2 = _ax.twinx()
_ax2.plot(MESES_C, _cm, 's--', color='#762a83', label='Factor C medio (Durigon)')
_ax2.axhline(cobertura_c['C_CVC_clase'].mean(), color='#d73027', ls=':', lw=1.6,
             label=f"C CVC por clase (media {cobertura_c['C_CVC_clase'].mean():.3f})")
_ax2.set_ylabel('Factor C', color='#762a83'); _ax2.set_ylim(0, 0.5)
_ax.set_title('Ciclo mensual de NDVI, EVI y Factor C (Sentinel-2 2017-2025, climatología)\n'
              'media de los 117 puntos', fontweight='bold')
_ln = _ax.get_lines() + _ax2.get_lines()
_ax.legend(_ln, [l.get_label() for l in _ln], loc='upper center',
           bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'Ciclo_mensual_NDVI_C.png', dpi=150, bbox_inches='tight')
plt.show()

fig, _axp = plt.subplots(figsize=(7.5, 6))
for _, _r in cobertura_c.iterrows():
    _axp.plot(MESES_C, _r[_cmc].values, color='#999999', alpha=.15, lw=.7)
_axp.plot(MESES_C, _cm, 'o-', color='#762a83', lw=2.5, label='media')
_axp.set_ylabel('Factor C mensual (Durigon 2014)'); _axp.set_ylim(0, 1)
_axp.set_title('Factor C mensual por punto (Sentinel-2, climatología 2017-2025)', fontweight='bold')
_axp.grid(alpha=.3); _axp.legend(); plt.tight_layout()
fig.savefig(DIR_FIGURAS / 'C_mensual_por_punto.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figuras: Ciclo_mensual_NDVI_C.png · C_mensual_por_punto.png')


## 13. Estadísticas descriptivas — resumen integrado de los factores base

> El **Factor K** (erodabilidad) y los factores **C** y **P** se desarrollan en la
> **Parte II** (Secciones 19–22), a partir de la base de suelos de la tesis. Esta
> sección resume únicamente los factores topográficos y de erosividad ya calculados.


In [ ]:
variables_resumen = [c for c in [
    'LS_Mitasova', 'LS_Clasico', 'Pendiente_grados', 'Pendiente_pct', 'FlowAccum',
    'Longitud_Ladera_m', 'P_anual', 'MFI', 'Factor_R', 'K',
] if c in puntos_gdf.columns]

tabla_resumen = puntos_gdf[variables_resumen].describe().T
display(tabla_resumen.round(3))


### 13‑A. Estadística percentílica completa — variables críticas

Resumen exigido (mínimo, máximo, media, mediana, desviación estándar y
percentiles P50/P75/P90/P95/P99) para las variables sometidas a auditoría
geomorfológica: longitud de ladera y factor LS.


In [ ]:
variables_a_resumir = []
if 'Longitud_Ladera_m' in puntos_gdf.columns:
    variables_a_resumir.append(resumen_percentiles_completo(
        puntos_gdf['Longitud_Ladera_m'], 'Longitud de ladera (m)'))
if 'LS_Mitasova' in puntos_gdf.columns:
    variables_a_resumir.append(resumen_percentiles_completo(
        puntos_gdf['LS_Mitasova'], 'Factor LS Mitasova'))
if 'LS_Clasico' in puntos_gdf.columns:
    variables_a_resumir.append(resumen_percentiles_completo(
        puntos_gdf['LS_Clasico'], 'Factor LS Clásico'))

if variables_a_resumir:
    tabla_percentiles_qc = pd.DataFrame(variables_a_resumir).set_index('variable')
    print('Estadística percentílica completa (P50, P75, P90, P95, P99):')
    display(tabla_percentiles_qc.round(2))
else:
    print('No se encontraron columnas de interés para el resumen percentílico.')


## 14. Mapa interactivo del Factor LS (Folium)

Mapa para la exploración espacial de la susceptibilidad topográfica a la erosión.
Círculos = puntos de muestreo; color y tamaño proporcionales a `LS_Clasico`.
El mapa se guarda como `Mapa_LS_Amaime_Umbral{STREAM_THRESHOLD}.html`.


In [ ]:
import folium
from folium.plugins import MeasureControl
import branca.colormap as cm

# 1. Reproyección a EPSG:4326 para Folium
puntos_mapa = puntos_gdf.to_crs('EPSG:4326').copy()

# 2. Escala de colores basada en LS_Clasico (respaldo: LS_Mitasova)
col_ls = 'LS_Clasico' if 'LS_Clasico' in puntos_mapa.columns else 'LS_Mitasova'
min_ls = puntos_mapa[col_ls].min()
max_ls = puntos_mapa[col_ls].quantile(0.95)

colormap = cm.LinearColormap(
    colors=['green', 'yellow', 'orange', 'red'], vmin=min_ls, vmax=max_ls,
    caption=f'Factor LS (umbral de drenaje: {STREAM_THRESHOLD} celdas)'
)

# 3. Mapa base centrado en los puntos
centro_lat = puntos_mapa.geometry.y.mean()
centro_lon = puntos_mapa.geometry.x.mean()
m = folium.Map(location=[centro_lat, centro_lon], zoom_start=12, control_scale=True)

# 4. Puntos
feature_group = folium.FeatureGroup(name=f'Puntos LS (umbral {STREAM_THRESHOLD})')
for idx, row in puntos_mapa.iterrows():
    punto_id = row.get('ID_UNAL', row.get('OBJECTID', idx))
    val_ls = row.get(col_ls, 0)
    val_r = row.get('Factor_R', 'N/A')

    popup_text = f"""
    <div style='font-family: Arial; font-size: 12px;'>
        <b>ID:</b> {punto_id}<br>
        <b>LS:</b> {val_ls:.3f}<br>
        <b>Pendiente (%):</b> {row.get('Pendiente_pct', 0):.2f}<br>
        <b>Long. ladera (m):</b> {row.get('Longitud_Ladera_m', 0):.1f}<br>
        <b>Factor R:</b> {val_r if isinstance(val_r, str) else f'{val_r:.2f}'}
    </div>
    """
    radius_val = 4 + (val_ls / max_ls * 10) if max_ls > 0 else 5

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=min(radius_val, 15), color='black', weight=0.5,
        fill=True, fill_color=colormap(val_ls), fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=250),
        tooltip=f'ID: {punto_id} | LS: {val_ls:.2f}',
    ).add_to(feature_group)

feature_group.add_to(m)
colormap.add_to(m)
folium.LayerControl().add_to(m)
m.add_child(MeasureControl())

ruta_html = DIR_MAPAS / f'Mapa_LS_Amaime_Umbral{STREAM_THRESHOLD}.html'
m.save(str(ruta_html))
print(f'Mapa exportado a: {ruta_html}')
display(m)


## 15. Exportación de los factores base

Punto de control: se exporta **`Cuenca_amaime_completo.xlsx`** (y una copia en
GeoPackage) con los factores topográficos y de erosividad, conservando todas las
columnas originales salvo las eliminadas por diseño (`LS`,
`Longitud_Ladera_RUSLE_m`). La tabla se **reexporta ampliada** al final de la
Parte II (Sección 28) con K, C, P, A y las variables de los cinco objetivos.


In [ ]:
columnas_originales = set(df_puntos.columns)
columnas_finales = set(puntos_gdf.drop(columns='geometry', errors='ignore').columns)
columnas_permitidas_borrar = {'LS', 'Longitud_Ladera_RUSLE_m'}

columnas_perdidas = (columnas_originales - columnas_finales) - columnas_permitidas_borrar
columnas_agregadas = columnas_finales - columnas_originales

if columnas_perdidas:
    raise AssertionError(f'Se perdieron columnas originales no autorizadas: {sorted(columnas_perdidas)}')

print(f'Columnas originales conservadas (o eliminadas por diseño): '
      f'{len(columnas_originales)}/{len(columnas_originales)}')
print(f'Columnas nuevas agregadas ({len(columnas_agregadas)}): {sorted(columnas_agregadas)}')

_ = exportar_resultados(
    puntos_gdf,
    ruta_xlsx=SALIDA_PUNTOS_XLSX,
    ruta_gpkg=SALIDA_PUNTOS_GPKG,
)
print(f'\nÉXITO: exportación intermedia (factores base) en {SALIDA_PUNTOS_XLSX.name} y {SALIDA_PUNTOS_GPKG.name}.')
display(puntos_gdf.drop(columns='geometry').head())


# Parte II — Erodabilidad, riesgo y pérdidas de suelo (Objetivos I–V)

Las secciones 16–29 desarrollan los cinco objetivos específicos:

* **Obj. I** (Sec. 19) — Caracterización físico‑química de los suelos.
* **Obj. II** (Sec. 20) — Factor **K** y su relación con las propiedades del suelo.
* **Obj. III** (Sec. 24) — Riesgo potencial de erosión hídrica, índice **LS × R**.
* **Obj. IV** (Sec. 25) — Pérdidas de suelo $A = R\cdot K\cdot LS\cdot C\cdot P$ y su distribución espacial.
* **Obj. V** (Sec. 26) — Variación temporal de $A$ 2010–2025 actualizando $R$.

Se integra la **base de la tesis** (`Cuenca Amaime Tesis Dayana.xlsx`, hoja
`Datos`), de donde se toman los **factores oficiales del modelo**: `LS Diego`,
`Factor K calculado`, `Factor R`, `Factor de Cultivos` (C) y `FACTOR P` (P). El
notebook recalcula cada factor como control y verifica que
$A = R\cdot K\cdot LS\cdot C\cdot P$ reproduce la columna `RUSLE (A)`. El área de
análisis es el **buffer de 5 km de los 117 puntos**, conforme al método.


## 16. Configuración y funciones de la extensión


In [ ]:
# --- Base de datos de suelos (tesis) ---
ARCHIVO_SUELOS = DIR_ENTRADA / 'Cuenca Amaime Tesis Dayana.xlsx'
HOJA_SUELOS    = 'Datos'
COL_JOIN       = 'ID_UNAL'

# --- Periodo del análisis temporal (Obj. V) ---
PERIODO_TEMP = (2010, 2025)

# --- Opciones del modelo ---
ENMASCARAR_CAUCES   = True            # excluye celdas de cauce (flow_acc > STREAM_THRESHOLD) del A ráster
FRAC_ARENA_MUY_FINA = 0.0             # fracción de arena tratada como "arena muy fina" en M (Wischmeier)
K_INTERP = 'idw'                      # espacialización de K a ráster: 'idw' (distancia inversa) o 'nn' (Voronoi)
IDW_POTENCIA, IDW_VECINOS = 2, 12     # parámetros de IDW
C_FUENTE = 'dayana'                   # 'dayana' (Factor de Cultivos de la base) o 'worldcover'
K_FUENTE = 'dayana'                   # 'dayana' = 'Factor K calculado' del estudio (Wischmeier con
                                     #   M, S de DMP y P de CH). Alternativas: 'wischmeier_si' | 'wischmeier_us'
K_MIN    = 0.0                        # sin piso: se respetan los valores de la base
LS_FUENTE_MODELO = 'dayana'           # 'dayana' = 'LS Diego' del estudio; 'mitasova' = recálculo del notebook

# --- Factor C por cobertura (ESA WorldCover v200 -> C) para el ráster ---
C_POR_COBERTURA = {
    10: 0.004,   # Árboles
    20: 0.020,   # Arbustos
    30: 0.050,   # Pastos / herbáceas
    40: 0.150,   # Cultivos
    50: 0.000,   # Urbano
    60: 0.450,   # Suelo desnudo / vegetación escasa
    70: 0.000,   # Nieve / hielo
    80: 0.000,   # Agua
    90: 0.010,   # Humedal herbáceo
    95: 0.004,   # Manglar
    100: 0.050,  # Musgo / liquen
}

# --- Clases de pérdida de suelo (FAO-PNUMA-UNESCO, 1980), t/ha/año ---
EROSION_BORDES   = [0, 10, 50, 200, 1e12]
EROSION_RANGOS   = ['0–10', '10–50', '50–200', '>200']
EROSION_ETIQUETAS = ['Ninguna o ligera', 'Moderada', 'Alta', 'Muy alta']
EROSION_COLORES  = ['#1a9850', '#fee08b', '#fc8d59', '#d73027']

# --- Niveles de riesgo potencial de erosión hídrica (índice LS·R, por quintiles) ---
RIESGO_ETIQUETAS = ['Muy bajo', 'Bajo', 'Moderado', 'Alto', 'Muy alto']
RIESGO_COLORES   = ['#1a9850', '#a6d96a', '#fee08b', '#fdae61', '#d73027']

# --- Superficie de cada píxel (ha) ---
AREA_PX_HA = (CELLSIZE_M ** 2) / 10_000.0

# --- Rutas de salida de la extensión ---
WORLDCOVER_FILE = DIR_RASTERS / 'ESA_WorldCover_Amaime.tif'
RIESGO_LSR_FILE = DIR_RASTERS / 'Riesgo_LSxR.tif'
RIESGO_LSR_CSV  = DIR_TABLAS / 'Resumen_riesgo_LSxR.csv'
PRIORIDAD_FILE  = DIR_RASTERS / 'Areas_prioritarias_conservacion.tif'
CARACTERIZACION_CSV = DIR_TABLAS / 'Caracterizacion_suelos.csv'
SINTESIS_CSV    = DIR_TABLAS / 'Sintesis_por_objetivo.csv'
PRECIP_MENSUAL_CSV = DIR_TABLAS / 'Precipitacion_mensual_por_punto.csv'
AGREGAR_HOJA_A_BASE_SUELOS = True    # añadir la hoja de precipitación mensual dentro de Cuenca Amaime Tesis Dayana.xlsx
FACTOR_K_FILE   = DIR_RASTERS / 'Factor_K.tif'
FACTOR_C_FILE   = DIR_RASTERS / 'Factor_C.tif'
FACTOR_P_FILE   = DIR_RASTERS / 'Factor_P.tif'
R_CLIM_FILE     = DIR_RASTERS / 'Factor_R_climatologico.tif'
R_PROM_FILE     = DIR_RASTERS / 'Factor_R_promedio_2010_2025.tif'
EE_R_ANUAL_FILE = DIR_RASTERS / 'CHIRPS_R_anual_2010_2025.tif'
A_CLIM_FILE     = DIR_RASTERS / 'A_USLE_climatologico.tif'
A_PROM_FILE     = DIR_RASTERS / 'A_USLE_promedio.tif'
A_STACK_FILE    = DIR_RASTERS / 'A_USLE_anual_2010_2025.tif'
TREND_A_FILE    = DIR_RASTERS / 'Tendencia_A_2010_2025.tif'
SERIE_TEMP_CSV  = DIR_TABLAS / 'Serie_temporal_R_A_2010_2025.csv'
RESUMEN_CLASES_CSV = DIR_TABLAS / 'Resumen_erosion_por_clase.csv'
CORR_K_CSV      = DIR_TABLAS / 'Correlacion_K_propiedades.csv'

print('Configuración de la extensión cargada. Periodo temporal:', PERIODO_TEMP)


In [ ]:
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from scipy.spatial import cKDTree
from scipy import stats


def perfil_grilla_ls():
    """Perfil rasterio de la malla de trabajo (la del Factor LS)."""
    with rasterio.open(LS_FILE) as s:
        return s.profile.copy()


def remuestrear_a_grilla(datos_src, perfil_src, perfil_dst, resampling=Resampling.bilinear):
    """Reproyecta/remuestrea un array 2D al CRS, transform y tamaño de perfil_dst."""
    destino = np.full((perfil_dst['height'], perfil_dst['width']), np.nan, dtype='float32')
    reproject(
        source=np.asarray(datos_src, dtype='float32'), destination=destino,
        src_transform=perfil_src['transform'], src_crs=perfil_src['crs'],
        dst_transform=perfil_dst['transform'], dst_crs=perfil_dst['crs'],
        src_nodata=np.nan, dst_nodata=np.nan, resampling=resampling,
    )
    return destino


def interpolar_idw(puntos_gdf, columna, perfil_dst, power=2, k=12):
    """Interpola una columna de puntos a la malla perfil_dst por IDW (k vecinos)."""
    g = puntos_gdf.dropna(subset=[columna]).to_crs(perfil_dst['crs'])
    xy = np.c_[g.geometry.x.values, g.geometry.y.values]
    val = g[columna].astype('float64').values
    tree = cKDTree(xy)

    h, w = perfil_dst['height'], perfil_dst['width']
    t = perfil_dst['transform']
    filas, cols = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    fr, cl = filas.ravel() + 0.5, cols.ravel() + 0.5
    xs = t.c + t.a * cl + t.b * fr
    ys = t.f + t.d * cl + t.e * fr
    dist, idx = tree.query(np.c_[xs, ys], k=min(k, len(val)))
    if dist.ndim == 1:
        dist, idx = dist[:, None], idx[:, None]
    dist = np.where(dist == 0, 1e-9, dist)
    wts = 1.0 / dist ** power
    out = np.sum(wts * val[idx], axis=1) / np.sum(wts, axis=1)
    return out.reshape(h, w).astype('float32')


def interpolar_nn(puntos_gdf, columna, perfil_dst):
    """Interpola una columna de puntos a la malla por vecino más cercano (tipo Voronoi):
    cada celda toma el valor del punto de muestreo más próximo. No suaviza."""
    g = puntos_gdf.dropna(subset=[columna]).to_crs(perfil_dst['crs'])
    tree = cKDTree(np.c_[g.geometry.x.values, g.geometry.y.values])
    val = g[columna].astype('float64').values

    h, w = perfil_dst['height'], perfil_dst['width']
    t = perfil_dst['transform']
    filas, cols = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    fr, cl = filas.ravel() + 0.5, cols.ravel() + 0.5
    xs = t.c + t.a * cl + t.b * fr
    ys = t.f + t.d * cl + t.e * fr
    _, idx = tree.query(np.c_[xs, ys], k=1)
    return val[idx].reshape(h, w).astype('float32')


def interpolar_a_grilla(puntos_gdf, columna, perfil_dst, metodo='nn'):
    """Envoltorio: 'nn' = vecino más cercano (Voronoi); 'idw' = distancia inversa ponderada."""
    if metodo == 'idw':
        return interpolar_idw(puntos_gdf, columna, perfil_dst,
                              power=IDW_POTENCIA, k=IDW_VECINOS)
    return interpolar_nn(puntos_gdf, columna, perfil_dst)


def mascara_cuenca(perfil_dst, zona_gdf):
    """Máscara booleana (True = dentro de la zona de influencia) sobre perfil_dst."""
    geoms = list(zona_gdf.to_crs(perfil_dst['crs']).geometry.values)
    fuera = geometry_mask(geoms, out_shape=(perfil_dst['height'], perfil_dst['width']),
                          transform=perfil_dst['transform'], invert=False)
    return ~fuera


def r_desde_precip_mensual(pmes):
    """Factor R (fórmula establecida) a partir de un stack (12, H, W) de precipitación mensual."""
    pmes = np.asarray(pmes, dtype='float64')
    p_anual = np.nansum(pmes, axis=0)
    sum_p2 = np.nansum(pmes ** 2, axis=0)
    with np.errstate(invalid='ignore', divide='ignore'):
        mfi = np.where(p_anual > 0, sum_p2 / p_anual, np.nan)
        r = np.where(mfi < 55,
                     0.7397 * mfi ** 1.847,
                     95.77 - 6.081 * mfi + 0.4770 * mfi ** 2)
    return r.astype('float32'), mfi.astype('float32')


def construir_r_anual_ee(region, y0, y1):
    """ee.Image con una banda de Factor R por año (R_<año>) para y0..y1, desde CHIRPS diario."""
    chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
              .filterDate(f'{y0}-01-01', f'{y1 + 1}-01-01')
              .filterBounds(region).select('precipitation'))

    def r_de_anio(y):
        y = ee.Number(y)
        meses = ee.List.sequence(1, 12)

        def suma_mes(m):
            m = ee.Number(m)
            return (chirps.filter(ee.Filter.calendarRange(y, y, 'year'))
                    .filter(ee.Filter.calendarRange(m, m, 'month')).sum())

        mensual = ee.ImageCollection(meses.map(suma_mes))
        p_anual = mensual.sum()
        sum_p2 = ee.ImageCollection(mensual.map(lambda im: im.pow(2))).sum()
        mfi = sum_p2.divide(p_anual)
        r = mfi.expression(
            'm < 55 ? 0.7397 * pow(m, 1.847) : 95.77 - 6.081 * m + 0.4770 * m * m', {'m': mfi})
        return r.rename(ee.String('R_').cat(y.format('%d')))

    anios = ee.List.sequence(y0, y1)
    return (ee.ImageCollection(anios.map(r_de_anio)).toBands()
            .rename([f'R_{y}' for y in range(y0, y1 + 1)]))


def clasificar_perdida_suelo(A):
    """Clasifica A (t/ha/año) según FAO-PNUMA-UNESCO (1980): 1=Ninguna/ligera .. 4=Muy alta."""
    A = np.asarray(A, dtype='float64')
    clase = np.full(A.shape, np.nan, dtype='float32')
    for i, (lo, hi) in enumerate(zip(EROSION_BORDES[:-1], EROSION_BORDES[1:]), start=1):
        clase = np.where((A >= lo) & (A < hi), i, clase)
    return clase


def resumen_por_clase(A, area_px_ha, mascara=None):
    """Tabla de ha, %, A media y pérdida (t/año) por clase de erosión, con fila TOTAL."""
    a = np.array(A, dtype='float64')
    if mascara is not None:
        a = np.where(mascara, a, np.nan)
    valid = np.isfinite(a)
    area_total = valid.sum() * area_px_ha
    filas = []
    for i, (lo, hi, rg, et) in enumerate(zip(EROSION_BORDES[:-1], EROSION_BORDES[1:],
                                             EROSION_RANGOS, EROSION_ETIQUETAS), start=1):
        m = valid & (a >= lo) & (a < hi)
        n = int(m.sum())
        ha = n * area_px_ha
        filas.append({'clase': i, 'etiqueta': et, 'rango_t_ha_ano': rg,
                      'area_ha': ha, 'area_pct': 100 * ha / area_total if area_total else np.nan,
                      'A_media_t_ha': float(np.nanmean(a[m])) if n else np.nan,
                      'perdida_t_ano': float(np.nansum(a[m]) * area_px_ha)})
    filas.append({'clase': 'TOTAL', 'etiqueta': '', 'rango_t_ha_ano': '',
                  'area_ha': area_total, 'area_pct': 100.0,
                  'A_media_t_ha': float(np.nanmean(a[valid])) if valid.any() else np.nan,
                  'perdida_t_ano': float(np.nansum(a[valid]) * area_px_ha)})
    return pd.DataFrame(filas)


def tendencia_mk_sen(anios, valores):
    """Mann-Kendall (tau, p) + pendiente de Sen (Theil-Sen) para una serie temporal."""
    anios = np.asarray(anios, dtype='float64')
    valores = np.asarray(valores, dtype='float64')
    ok = np.isfinite(valores)
    tau, p = stats.kendalltau(anios[ok], valores[ok])
    sen, icpt, lo, hi = stats.theilslopes(valores[ok], anios[ok])
    if p < 0.05:
        tendencia = 'creciente significativa' if sen > 0 else 'decreciente significativa'
    else:
        tendencia = 'sin tendencia significativa'
    return {'kendall_tau': tau, 'p_value': p, 'sen_slope': sen, 'sen_intercept': icpt,
            'sen_lo': lo, 'sen_hi': hi, 'tendencia': tendencia}


def pendiente_ols_pixel(stack, anios):
    """Pendiente lineal (OLS) por píxel de un stack (T, H, W) frente al vector `anios`."""
    t = np.asarray(anios, dtype='float64')
    t = t - t.mean()
    y = np.asarray(stack, dtype='float64')
    num = np.nansum(t[:, None, None] * (y - np.nanmean(y, axis=0)[None]), axis=0)
    den = np.nansum(t ** 2) + 1e-12
    return (num / den).astype('float32')


print('Funciones de la extensión cargadas.')


## 17. Integración de la base de datos de suelos

Se une la hoja `Datos` de `Cuenca Amaime Tesis Dayana.xlsx` a la tabla de puntos
por `ID_UNAL` (117/117) y se traen las variables edáficas (textura A/L/AR,
materia orgánica, densidad aparente, código de estructura, código de
permeabilidad), la unidad cartográfica de suelo y los factores **C y P** del
estudio de cobertura/uso.


In [ ]:
_sue = pd.read_excel(ARCHIVO_SUELOS, sheet_name=HOJA_SUELOS)
_ccol = [c for c in _sue.columns if c.startswith('Factor de Cultivos')][0]
_kcol = [c for c in _sue.columns if 'Factor K calculado' in c][0]
_cod  = [c for c in _sue.columns if 'digo Suelo 2004' in c]
_lscol = [c for c in _sue.columns if c.strip() == 'LS Diego']
_llcol = [c for c in _sue.columns if 'Longitud ladera Diego' in c]

# Se traen de la base los FACTORES OFICIALES del estudio (los que están en el Word):
#   LS = 'LS Diego', K = 'Factor K calculado', R = 'Factor R',
#   C = 'Factor de Cultivos', P = 'FACTOR P', A = 'RUSLE (A)'.
_ren = {
    'A': 'Arena_pct', 'L': 'Limo_pct', 'AR': 'Arcilla_pct', 'TEXTURA': 'Textura_USDA',
    'MO': 'MO_pct', 'DA': 'Da_gcm3', 'S.1': 'Estructura_cod', 'PERMEA': 'Permeabilidad_cod',
    _kcol: 'K_Dayana', _ccol: 'C_Dayana', 'FACTOR P': 'P_Dayana',
    'Factor R': 'R_Dayana', 'RUSLE (A)': 'A_Dayana',
    'Nombre Suelo 2004': 'Nombre_Suelo', 'Orden Suelo': 'Orden_Suelo',
}
if _cod:
    _ren[_cod[0]] = 'Cod_Suelo'
if _lscol:
    _ren[_lscol[0]] = 'LS_Dayana'
if _llcol:
    _ren[_llcol[0]] = 'LongLadera_Dayana'

_sub = _sue[[COL_JOIN] + list(_ren)].rename(columns=_ren)
for c in ['Arena_pct', 'Limo_pct', 'Arcilla_pct', 'MO_pct', 'Da_gcm3', 'Estructura_cod',
          'Permeabilidad_cod', 'K_Dayana', 'C_Dayana', 'P_Dayana', 'R_Dayana', 'A_Dayana',
          'LS_Dayana', 'LongLadera_Dayana']:
    if c in _sub.columns:
        _sub[c] = pd.to_numeric(_sub[c], errors='coerce')

puntos_gdf = gpd.GeoDataFrame(
    puntos_gdf.merge(_sub, on=COL_JOIN, how='left', validate='one_to_one'),
    geometry='geometry', crs=puntos_gdf.crs)

# LS oficial del modelo por punto: 'LS Diego' del estudio (o el recálculo Mitasova)
if LS_FUENTE_MODELO == 'dayana' and 'LS_Dayana' in puntos_gdf.columns:
    puntos_gdf['LS_oficial'] = puntos_gdf['LS_Dayana']
    _ls_src = "'LS Diego' del estudio"
else:
    puntos_gdf['LS_oficial'] = np.nan   # se rellena en la Sección 11 con LS_Mitasova
    _ls_src = 'recálculo Mitasova (Sección 11)'

print(f'Base de suelos integrada: {_sub.shape[1] - 1} variables, '
      f'{puntos_gdf[["Arena_pct", "MO_pct", "K_Dayana"]].notna().all(axis=1).sum()}/117 puntos completos.')
print(f'LS oficial del modelo: {_ls_src}')
display(puntos_gdf[['ID_UNAL', 'Arena_pct', 'Limo_pct', 'Arcilla_pct', 'Textura_USDA', 'MO_pct',
                    'Estructura_cod', 'Permeabilidad_cod', 'LS_Dayana', 'K_Dayana', 'C_Dayana',
                    'P_Dayana', 'R_Dayana', 'A_Dayana']].head())


## 18. Química complementaria de la base de suelos

Se incorporan al `GeoDataFrame` algunas variables químicas adicionales de la base
(pH, conductividad eléctrica, CICE, bases intercambiables, nitrógeno total) para
la caracterización del Objetivo I.


In [ ]:
_extra = pd.read_excel(ARCHIVO_SUELOS, sheet_name=HOJA_SUELOS)
_map_qui = {'PH': 'pH', 'CE': 'CE_dSm', 'CICE': 'CICE_cmol', 'CICA': 'CICA_cmol',
            'CA': 'Ca_cmol', 'MG': 'Mg_cmol', 'K': 'K_cmol', 'NT': 'N_total_pct',
            'AL': 'Al_cmol', 'P': 'P_mg_kg'}
for _src, _dst in _map_qui.items():
    if _src in _extra.columns and _dst not in puntos_gdf.columns:
        _serie = pd.to_numeric(_extra.set_index(COL_JOIN)[_src], errors='coerce')
        puntos_gdf[_dst] = puntos_gdf[COL_JOIN].map(_serie)
_qui_ok = [d for d in _map_qui.values() if d in puntos_gdf.columns and puntos_gdf[d].notna().any()]
print('Variables químicas incorporadas:', _qui_ok)


## 19. Objetivo I — Caracterización físico‑química de los suelos

Resumen estadístico de las propiedades físicas (textura, densidad aparente,
estructura, permeabilidad) y químicas (pH, CE, materia orgánica, CICE, bases) de
los suelos de la cuenca, y su distribución por **clase textural** y por **orden
taxonómico de suelo**.


In [ ]:
_fis = [c for c in ['Arena_pct', 'Limo_pct', 'Arcilla_pct', 'MO_pct', 'Da_gcm3',
                    'Estructura_cod', 'Permeabilidad_cod'] if c in puntos_gdf.columns]
_qui = [c for c in ['pH', 'CE_dSm', 'CICE_cmol', 'Ca_cmol', 'Mg_cmol', 'K_cmol',
                    'N_total_pct', 'P_mg_kg'] if c in puntos_gdf.columns]

tabla_caracterizacion = (puntos_gdf[_fis + _qui]
                         .describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])
tabla_caracterizacion.to_csv(CARACTERIZACION_CSV)
print('=== Obj. I — Caracterización físico-química de los suelos (n = 117) ===')
display(tabla_caracterizacion.round(2))

print('\nDistribución por clase textural (USDA):')
display(puntos_gdf['Textura_USDA'].value_counts().rename('n_puntos').to_frame())
if 'Orden_Suelo' in puntos_gdf.columns:
    print('Distribución por orden de suelo:')
    display(puntos_gdf['Orden_Suelo'].value_counts().rename('n_puntos').to_frame())


In [ ]:
# Histogramas de las propiedades clave y relación textura-erodabilidad
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, v, u in zip(axes, ['Arcilla_pct', 'Limo_pct', 'MO_pct', 'Da_gcm3'],
                    ['%', '%', '%', 'g/cm³']):
    puntos_gdf[v].plot.hist(ax=ax, bins=18, color='#8c6d4f', edgecolor='white')
    ax.set_xlabel(f'{v} ({u})'); ax.set_ylabel('n puntos')
fig.suptitle('Objetivo I — Distribución de propiedades del suelo', fontweight='bold')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5))
_sc = ax.scatter(puntos_gdf['Arena_pct'], puntos_gdf['Arcilla_pct'],
                 c=puntos_gdf['MO_pct'], cmap='YlGn', s=32, edgecolor='0.4', linewidth=.3)
ax.set_xlabel('% Arena'); ax.set_ylabel('% Arcilla')
ax.set_title('Composición textural de los puntos (color = MO %)', fontweight='bold')
fig.colorbar(_sc, ax=ax, label='Materia orgánica (%)'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 20. Objetivo II — Factor K y su relación con las propiedades del suelo

El factor **K** oficial del modelo es la columna **`Factor K calculado`** del
estudio (`K_Dayana`), obtenida con la ecuación de Wischmeier & Smith (1978) a
partir del parámetro textural **M**, la estructura **S** (derivada del DMP) y la
permeabilidad **P** (derivada de la conductividad hidráulica), tal como se
documenta en la metodología (`DETALLE RESULTADOS.docx`). Como control interno se
recalcula K con la misma ecuación a partir de textura + MO + estructura +
permeabilidad y se compara. Luego se analiza la **relación de K con las
propiedades** del suelo (correlación, regresión múltiple, K por clase textural y
orden de suelo).


In [ ]:
# K oficial del modelo = 'Factor K calculado' del estudio (K_FUENTE = 'dayana')
_fuentes_k = {
    'dayana':        'K_Dayana',
    'wischmeier_si': 'K_Wischmeier_SI',
    'wischmeier_us': 'K_Wischmeier',
}

# Control interno: recálculo de K con la ecuación de Wischmeier & Smith (1978)
_amf = FRAC_ARENA_MUY_FINA * puntos_gdf['Arena_pct']
puntos_gdf['K_Wischmeier'] = calcular_factor_k_wischmeier(
    puntos_gdf['MO_pct'], puntos_gdf['Limo_pct'], _amf, puntos_gdf['Arcilla_pct'],
    puntos_gdf['Estructura_cod'], puntos_gdf['Permeabilidad_cod'], convertir_si=False)
puntos_gdf['K_Wischmeier_SI'] = puntos_gdf['K_Wischmeier'] * 0.1317

puntos_gdf['K_factor'] = puntos_gdf[_fuentes_k[K_FUENTE]]
if K_MIN:
    puntos_gdf['K_factor'] = puntos_gdf['K_factor'].clip(lower=K_MIN)

display(puntos_gdf[['K_Dayana', 'K_Wischmeier', 'K_Wischmeier_SI']].describe().T.round(4))
_r = puntos_gdf[['K_Dayana', 'K_Wischmeier']].corr().iloc[0, 1]
print(f'Fuente de K del modelo: {K_FUENTE}  ->  K medio = {puntos_gdf["K_factor"].mean():.4f}  '
      f'(rango {puntos_gdf["K_factor"].min():.4f}–{puntos_gdf["K_factor"].max():.4f})')
print(f'Correlación K_Dayana vs recálculo Wischmeier: r = {_r:.3f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(puntos_gdf['K_Wischmeier'], puntos_gdf['K_Dayana'], s=18, alpha=.6)
ax.set_xlabel('K Wischmeier (control)'); ax.set_ylabel('K del estudio (oficial)')
ax.set_title('K del estudio vs recálculo de control'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Objetivo II: relación del Factor K con las propiedades físico-químicas del suelo
_props = ['Arcilla_pct', 'Limo_pct', 'Arena_pct', 'MO_pct', 'Da_gcm3', 'Estructura_cod', 'Pendiente_pct']
_props = [p for p in _props if p in puntos_gdf.columns]
corr_k = puntos_gdf[['K_factor'] + _props].corr()['K_factor'].drop('K_factor').to_frame('r_Pearson')
corr_k['r_Spearman'] = [puntos_gdf[['K_factor', p]].corr(method='spearman').iloc[0, 1] for p in corr_k.index]
corr_k = corr_k.reindex(corr_k['r_Pearson'].abs().sort_values(ascending=False).index)
print('=== Obj. II — Correlación del Factor K con las propiedades del suelo ===')
display(corr_k.round(3))
corr_k.to_csv(CORR_K_CSV)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, p in zip(axes, ['Arcilla_pct', 'Limo_pct', 'MO_pct']):
    ax.scatter(puntos_gdf[p], puntos_gdf['K_factor'], s=16, alpha=.6)
    ax.set_xlabel(p); ax.set_ylabel('K'); ax.grid(alpha=.3)
axes[0].set_title('Factor K frente a las propiedades del suelo')
plt.tight_layout(); plt.show()


In [ ]:
# Regresión lineal múltiple  K ~ propiedades  +  K por clase textural / orden de suelo
_Xc = [c for c in ['Arcilla_pct', 'Limo_pct', 'Arena_pct', 'MO_pct', 'Da_gcm3'] if c in puntos_gdf.columns]
_d = puntos_gdf[['K_factor'] + _Xc].dropna()
_X = np.c_[np.ones(len(_d)), _d[_Xc].values]
_beta, *_ = np.linalg.lstsq(_X, _d['K_factor'].values, rcond=None)
_yhat = _X @ _beta
_r2 = 1 - np.sum((_d['K_factor'].values - _yhat) ** 2) / np.sum((_d['K_factor'].values - _d['K_factor'].mean()) ** 2)
print('=== Obj. II — Regresión lineal múltiple:  K ~ ' + ' + '.join(_Xc) + ' ===')
for _n, _b in zip(['(intercepto)'] + _Xc, _beta):
    print(f'  {_n:16s}: {_b:+.6f}')
print(f'  R² = {_r2:.3f}   (n = {len(_d)})')

print('\nK medio por clase textural (USDA):')
display(puntos_gdf.groupby('Textura_USDA')['K_factor'].agg(['mean', 'std', 'count']).round(4))
if 'Orden_Suelo' in puntos_gdf.columns:
    print('K medio por orden de suelo:')
    display(puntos_gdf.groupby('Orden_Suelo')['K_factor'].agg(['mean', 'std', 'count']).round(4))


## 21. Factor C — Cobertura y uso del suelo

Para el **modelo por punto** se usa el factor C del estudio edafológico
(`C_Dayana`, columna *Factor de Cultivos*). Para el **ráster** se usa la tabla
`C_POR_COBERTURA` aplicada a **ESA WorldCover v200** (10 m, agregado a 30 m),
descargado desde Earth Engine.


In [ ]:
perfil_ls = perfil_grilla_ls()

if FORZAR_DESCARGA_EE or not WORLDCOVER_FILE.exists():
    _wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(AOI)
    _ee_descargar_geotiff(_wc, AOI, WORLDCOVER_FILE, scale=30, crs=CRS_PROYECTADO,
                          nombre='ESA WorldCover v200')
else:
    print(f'WorldCover ya presente: {WORLDCOVER_FILE.name}')

_wc_arr, _wc_perf, _ = leer_raster(WORLDCOVER_FILE)
wc_grid = remuestrear_a_grilla(_wc_arr, _wc_perf, perfil_ls, resampling=Resampling.nearest)

C_worldcover = np.full(wc_grid.shape, np.nan, dtype='float32')
for _cod_wc, _cval in C_POR_COBERTURA.items():
    C_worldcover = np.where(np.round(wc_grid) == _cod_wc, _cval, C_worldcover)

# Factor C por punto
puntos_gdf['C_factor'] = puntos_gdf['C_Dayana'] if C_FUENTE == 'dayana' else np.nan
print('C por punto (fuente =', C_FUENTE, '):')
print(puntos_gdf['C_factor'].describe().round(3).to_dict())
print('C ráster (WorldCover):',
      pd.Series(C_worldcover[np.isfinite(C_worldcover)]).describe().round(3).to_dict())


## 22. Factor P — Prácticas de conservación

Para el **modelo por punto** se usa el factor P del estudio (`P_Dayana`, de la
capa de cobertura/uso CVC). Para el **ráster** se adopta el escenario base
**P = 1** (sin prácticas de conservación cartografiadas a escala de cuenca).


In [ ]:
puntos_gdf['P_factor'] = puntos_gdf['P_Dayana'].fillna(1.0)
mask_cuenca = mascara_cuenca(perfil_ls, zona_gdf)
P_raster = np.where(mask_cuenca, 1.0, np.nan).astype('float32')

print('P por punto (base tesis):', puntos_gdf['P_factor'].describe().round(3).to_dict())
print(f'P ráster: 1.0 uniforme dentro de la cuenca ({int(mask_cuenca.sum()):,} celdas).')


## 23. Rásteres de factores alineados a la malla de 30 m

Todos los factores se llevan a la malla del Factor LS (EPSG:3115, 30 m):

* **R** — climatológico (climatología mensual CHIRPS ya descargada) y **anual
  2010–2025** (descarga de un stack de 16 bandas desde Earth Engine), remuestreados
  de ~5,5 km a 30 m.
* **K** — espacialización del K por punto (base de la tesis) por **vecino más
  cercano** (Voronoi): cada celda toma el K del punto de muestreo más próximo, sin
  suavizado (`K_INTERP`; alternativa `'idw'`).
* **C** — tabla por cobertura (WorldCover).
* **P** — escenario base = 1.
* **LS** — el ráster de Mitasova; opcionalmente se excluyen las celdas de cauce.


In [ ]:
# --- R climatológico (desde la climatología mensual CHIRPS) ---
with rasterio.open(CHIRPS_MENSUAL_FILE) as s:
    _pmes_clim = s.read().astype('float64')          # (12, h, w)
    _pmes_perf = s.profile.copy()
R_clim_native, _ = r_desde_precip_mensual(_pmes_clim)
_perf_1b = {**_pmes_perf, 'count': 1}
R_clim = remuestrear_a_grilla(R_clim_native, _perf_1b, perfil_ls)

# --- R anual 2010-2025 (stack de 16 bandas desde Earth Engine) ---
y0, y1 = PERIODO_TEMP
anios = list(range(y0, y1 + 1))
if FORZAR_DESCARGA_EE or not EE_R_ANUAL_FILE.exists():
    _ee_descargar_geotiff(construir_r_anual_ee(AOI, y0, y1), AOI, EE_R_ANUAL_FILE,
                          scale=5566, crs=CRS_GEOGRAFICO, nombre=f'R anual {y0}-{y1}')
else:
    print(f'R anual ya presente: {EE_R_ANUAL_FILE.name}')

with rasterio.open(EE_R_ANUAL_FILE) as s:
    _r_anual_native = s.read().astype('float32')     # (16, h, w)
    _r_anual_perf = {**s.profile.copy(), 'count': 1}
R_anual = np.stack([remuestrear_a_grilla(_r_anual_native[i], _r_anual_perf, perfil_ls)
                    for i in range(len(anios))]).astype('float32')   # (16, H, W)
R_prom = np.nanmean(R_anual, axis=0).astype('float32')

for _nom, _arr, _ruta in [('R climatológico', R_clim, R_CLIM_FILE),
                          ('R promedio 2010-2025', R_prom, R_PROM_FILE)]:
    guardar_raster(_ruta, np.where(mask_cuenca, _arr, np.nan), perfil_ls)
    print(f'{_nom}: media de cuenca = {np.nanmean(np.where(mask_cuenca, _arr, np.nan)):.0f}')


In [ ]:
# --- K, C, P, LS alineados a la malla ---
K_raster = interpolar_a_grilla(puntos_gdf, 'K_factor', perfil_ls, metodo=K_INTERP)
print(f'K espacializado por: {"vecino más cercano (Voronoi)" if K_INTERP == "nn" else "IDW"}')
K_raster = np.where(mask_cuenca, K_raster, np.nan).astype('float32')

C_raster = np.where(mask_cuenca, C_worldcover, np.nan).astype('float32')

LS_sel, _, _ = leer_raster(LS_FILE)          # el mapa de A usa siempre LS Mitasova (único LS ráster)
LS_raster = np.where(mask_cuenca, LS_sel, np.nan).astype('float32')

if ENMASCARAR_CAUCES:
    _cauce = np.isfinite(flow_acc) & (flow_acc > STREAM_THRESHOLD)
    LS_raster = np.where(_cauce, np.nan, LS_raster)
    print(f'Celdas de cauce excluidas del mapa de A: {int(np.nansum(_cauce)):,}')

for _nom, _arr, _ruta in [('K', K_raster, FACTOR_K_FILE), ('C', C_raster, FACTOR_C_FILE),
                          ('P', P_raster, FACTOR_P_FILE)]:
    guardar_raster(_ruta, _arr, perfil_ls)
    _v = _arr[np.isfinite(_arr)]
    print(f'Factor {_nom}: media={_v.mean():.4f}  min={_v.min():.4f}  max={_v.max():.4f}')
print('Rásteres de factores alineados a', perfil_ls['width'], 'x', perfil_ls['height'], 'px.')


## 24. Objetivo III — Riesgo potencial de erosión hídrica (índice LS × R)

El **riesgo potencial** de erosión hídrica se evalúa con el índice $LS \times R$,
que combina la susceptibilidad topográfica (LS) con la agresividad de la lluvia
(R) **sin** intervenir la erodabilidad del suelo (K) ni el manejo (C, P) — es
decir, el riesgo inherente del terreno y el clima. Se calcula por ráster (30 m,
$R$ promedio 2010–2025) y por punto, se clasifica en **cinco niveles por
quintiles** y se cartografía.


In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# Índice LS x R (ráster). LS_raster ya viene enmascarado (cuenca y, opcionalmente, sin cauces).
LSR_raster = (LS_raster * R_prom).astype('float32')
guardar_raster(RIESGO_LSR_FILE, np.where(mask_cuenca, LSR_raster, np.nan), perfil_ls)

# Clasificación en 5 niveles por quintiles del índice
_v = LSR_raster[np.isfinite(LSR_raster)]
_q = np.nanpercentile(_v, [20, 40, 60, 80])
_bordes_riesgo = [-np.inf, *_q, np.inf]


def clasificar_riesgo(x):
    x = np.asarray(x, dtype='float64')
    c = np.full(x.shape, np.nan, dtype='float32')
    for i in range(5):
        c = np.where((x > _bordes_riesgo[i]) & (x <= _bordes_riesgo[i + 1]), i + 1, c)
    return c


LSR_clase = clasificar_riesgo(LSR_raster)
puntos_gdf['LS_x_R'] = puntos_gdf['LS_oficial'] * puntos_gdf['Factor_R']
puntos_gdf['Riesgo_LSxR'] = clasificar_riesgo(puntos_gdf['LS_x_R'].values)

_n_val = int(np.isfinite(LSR_raster).sum())
_filas = []
for i, et in enumerate(RIESGO_ETIQUETAS, start=1):
    _m = LSR_clase == i
    _filas.append({'nivel': i, 'riesgo': et,
                   'area_ha': int(_m.sum()) * AREA_PX_HA,
                   'area_pct': 100 * int(_m.sum()) / _n_val,
                   'LSxR_medio': float(np.nanmean(LSR_raster[_m])) if _m.any() else np.nan})
resumen_riesgo = pd.DataFrame(_filas)
resumen_riesgo.to_csv(RIESGO_LSR_CSV, index=False)
print('=== Obj. III — Riesgo potencial de erosión hídrica (índice LS x R, R promedio 2010-2025) ===')
display(resumen_riesgo.round(2))
print('Puntos por nivel de riesgo:')
display(puntos_gdf['Riesgo_LSxR'].map(dict(enumerate(RIESGO_ETIQUETAS, start=1)))
        .value_counts().rename('n_puntos').to_frame())


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 7))
with np.errstate(invalid='ignore', divide='ignore'):
    _logLSR = np.log10(np.clip(np.where(mask_cuenca, LSR_raster, np.nan), 1, None))
_im = ax[0].imshow(_logLSR, cmap='magma')
ax[0].set_title('log10(LS × R)', fontweight='bold'); ax[0].axis('off')
fig.colorbar(_im, ax=ax[0], fraction=0.046, pad=0.04, shrink=.85)

ax[1].imshow(LSR_clase, cmap=ListedColormap(RIESGO_COLORES), vmin=.5, vmax=5.5)
ax[1].set_title('Niveles de riesgo potencial (LS × R)', fontweight='bold'); ax[1].axis('off')
ax[1].legend(handles=[mpatches.Patch(color=c, label=l) for c, l in zip(RIESGO_COLORES, RIESGO_ETIQUETAS)],
             loc='center left', bbox_to_anchor=(1.02, .5), frameon=False, title='Riesgo')
plt.tight_layout(rect=[0, 0, .95, 1]); plt.show()


## 25. Objetivo IV — Pérdidas de suelo A = R · K · LS · C · P

Se calcula $A$ con dos versiones del factor R (climatológico 2020–2025 y promedio
2010–2025), se clasifica según **FAO‑PNUMA‑UNESCO (1980)** y se reporta el área
(ha) y la pérdida (t/año) por clase. Además se calcula $A$ **por punto** con los
datos reales de suelo y se compara con el $A$ de la base de la tesis.

> **Lectura de las magnitudes.** El factor R usa la ecuación MFI→R de
> Renard & Freimund (1994), que en climas húmedos tropicales tiende a valores
> altos (aquí R ≈ 5 000–8 000). Los valores de $A$ dependen fuertemente de R y del
> LS de Mitasova (no acotado en celdas de convergencia); conviene interpretarlos
> en términos **relativos / de clases** y, de ser posible, validar R contra
> $EI_{30}$ de pluviógrafos locales.


In [ ]:
AREA_PX_HA = (CELLSIZE_M ** 2) / 10_000.0


def calcular_A(R_arr):
    return (R_arr * K_raster * LS_raster * C_raster * P_raster).astype('float32')


A_clim = calcular_A(R_clim)
A_prom = calcular_A(R_prom)
guardar_raster(A_CLIM_FILE, A_clim, perfil_ls)
guardar_raster(A_PROM_FILE, A_prom, perfil_ls)

# A oficial del modelo = con el R climatológico (el mismo 'Factor R' del estudio).
# A_prom (R medio 2010-2025) se reserva para el análisis temporal (Obj. V).
A_modelo = A_clim

for _nom, _A in [('A (R climatológico 2020-2025)', A_clim), ('A (R promedio 2010-2025)', A_prom)]:
    _a = _A[np.isfinite(_A)]
    print(f'{_nom}:  media={_a.mean():.1f}  mediana={np.median(_a):.1f}  '
          f'P90={np.percentile(_a, 90):.1f}  máx={_a.max():.1f} t/ha/año  |  '
          f'pérdida total = {np.nansum(_A) * AREA_PX_HA:,.0f} t/año')


In [ ]:
resumen = resumen_por_clase(A_modelo, AREA_PX_HA)
resumen.to_csv(RESUMEN_CLASES_CSV, index=False)
print('=== Obj. IV — Distribución de la pérdida de suelo por clase (R climatológico) ===')
display(resumen.round(2))

A_clase = clasificar_perdida_suelo(A_modelo)
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
with np.errstate(invalid='ignore', divide='ignore'):
    _logA = np.log10(np.clip(A_modelo, 0.1, None))
im0 = axes[0].imshow(_logA, cmap='inferno')
axes[0].set_title('log10(A)  —  pérdida de suelo (t/ha/año)', fontweight='bold'); axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, shrink=.85)

axes[1].imshow(A_clase, cmap=ListedColormap(EROSION_COLORES), vmin=0.5, vmax=4.5)
axes[1].set_title('Clases de erosión (FAO-PNUMA-UNESCO, 1980)', fontweight='bold'); axes[1].axis('off')
axes[1].legend(handles=[mpatches.Patch(color=c, label=f'{l} ({r})')
                        for c, l, r in zip(EROSION_COLORES, EROSION_ETIQUETAS, EROSION_RANGOS)],
               loc='center left', bbox_to_anchor=(1.02, .5), frameon=False)
plt.tight_layout(rect=[0, 0, .95, 1]); plt.show()


In [ ]:
# --- A por punto = R · K · LS · C · P con los factores oficiales del estudio ---
puntos_gdf['R_factor'] = puntos_gdf['Factor_R']          # R de CHIRPS (== 'Factor R' del estudio)
if puntos_gdf['LS_oficial'].isna().all():
    puntos_gdf['LS_oficial'] = puntos_gdf['LS_Mitasova']

puntos_gdf['A_USLE_punto'] = (puntos_gdf['R_factor'] * puntos_gdf['K_factor']
                              * puntos_gdf['LS_oficial'] * puntos_gdf['C_factor']
                              * puntos_gdf['P_factor'])

# Verificación: debe reproducir la columna 'RUSLE (A)' del estudio
if 'A_Dayana' in puntos_gdf.columns:
    _d = (puntos_gdf['A_USLE_punto'] - puntos_gdf['A_Dayana']).abs()
    _rel = (_d / puntos_gdf['A_Dayana'].replace(0, np.nan)).max()
    print(f"Verificación A_USLE_punto vs 'RUSLE (A)' del estudio:")
    print(f"  diferencia absoluta máx = {_d.max():.4g} t/ha/año  |  relativa máx = {_rel*100:.3f} %")
    print('  -> COINCIDE' if _d.max() < 0.5 else '  -> revisar (no coincide dentro de 0.5 t/ha/año)')
    _rR = puntos_gdf[['Factor_R', 'R_Dayana']].corr().iloc[0, 1]
    print(f"  (R de CHIRPS vs 'Factor R' del estudio: r = {_rR:.4f})")

guardar_raster(DIR_RASTERS / '_tmp_A_prom.tif', A_modelo, perfil_ls)
puntos_gdf = extraer_valor_raster_puntos(puntos_gdf, DIR_RASTERS / '_tmp_A_prom.tif', 'A_USLE_raster')
(DIR_RASTERS / '_tmp_A_prom.tif').unlink()

puntos_gdf['Clase_erosion'] = clasificar_perdida_suelo(puntos_gdf['A_USLE_punto'].values)
display(puntos_gdf[['A_USLE_punto', 'A_Dayana', 'A_USLE_raster']].describe().T.round(1))
print('\nDistribución por clase de erosión (A por punto):')
print(puntos_gdf['Clase_erosion'].map(dict(enumerate(EROSION_ETIQUETAS, start=1))).value_counts().to_string())


## 26. Objetivo V — Variación temporal de las pérdidas 2010–2025

Se calcula $A$ para cada año actualizando **solo el factor R** (K, LS, C y P se
mantienen constantes, conforme al objetivo). Se analiza la tendencia de la media
de la cuenca con **Mann‑Kendall** (τ, p) y **pendiente de Sen**, se estima la
pendiente lineal por píxel y se cartografía el cambio entre la primera y la
segunda mitad del periodo.


In [ ]:
# A anual (solo varía R)
A_anual = np.stack([calcular_A(R_anual[i]) for i in range(len(anios))]).astype('float32')  # (16, H, W)

serie = []
for i, y in enumerate(anios):
    _Ay = A_anual[i]
    _Ry = np.where(mask_cuenca, R_anual[i], np.nan)
    serie.append({'anio': y,
                  'R_medio': float(np.nanmean(_Ry)),
                  'A_medio_t_ha': float(np.nanmean(_Ay)),
                  'perdida_total_t_ano': float(np.nansum(_Ay) * AREA_PX_HA)})
serie = pd.DataFrame(serie)
serie.to_csv(SERIE_TEMP_CSV, index=False)
display(serie.round(2))


In [ ]:
mk_R = tendencia_mk_sen(serie['anio'], serie['R_medio'])
mk_A = tendencia_mk_sen(serie['anio'], serie['A_medio_t_ha'])
print('=== Obj. V — Tendencia 2010-2025 (media de la cuenca) ===')
print(f"  R: Sen = {mk_R['sen_slope']:+.1f} por año | tau = {mk_R['kendall_tau']:+.2f} | "
      f"p = {mk_R['p_value']:.3f}  ->  {mk_R['tendencia']}")
print(f"  A: Sen = {mk_A['sen_slope']:+.3f} t/ha/año por año | tau = {mk_A['kendall_tau']:+.2f} | "
      f"p = {mk_A['p_value']:.3f}  ->  {mk_A['tendencia']}")
_camb_pct = 100 * (serie['A_medio_t_ha'].iloc[-1] - serie['A_medio_t_ha'].iloc[0]) / serie['A_medio_t_ha'].iloc[0]
print(f"  Cambio total A (primer vs último año): {_camb_pct:+.1f}%")

# Pendiente lineal por píxel + stack anual
pend_A = np.where(mask_cuenca, pendiente_ols_pixel(A_anual, anios), np.nan)
guardar_raster(TREND_A_FILE, pend_A, perfil_ls)

_perf_stack = perfil_ls.copy()
_perf_stack.update(count=len(anios), dtype='float32', nodata=np.nan)
with rasterio.open(A_STACK_FILE, 'w', **_perf_stack) as dst:
    for i, y in enumerate(anios):
        dst.write(A_anual[i], i + 1)
        dst.set_band_description(i + 1, f'A_{y}')
print(f'Guardados: {TREND_A_FILE.name} y {A_STACK_FILE.name} (16 bandas).')


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(serie['anio'], serie['A_medio_t_ha'], 'o-', color='#d73027')
ax[0].plot(serie['anio'], mk_A['sen_intercept'] + mk_A['sen_slope'] * serie['anio'], '--', color='k',
           label=f"Sen: {mk_A['sen_slope']:+.3f} t/ha/año·año")
ax[0].set_title('Pérdida de suelo media de la cuenca'); ax[0].set_ylabel('A (t/ha/año)')
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(serie['anio'], serie['R_medio'], 's-', color='#4575b4')
ax[1].set_title('Erosividad media de la cuenca (R)'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

_mitad = len(anios) // 2
_camb = np.where(mask_cuenca, np.nanmean(A_anual[_mitad:], axis=0) - np.nanmean(A_anual[:_mitad], axis=0), np.nan)
_vmax = np.nanpercentile(np.abs(_camb), 98)
fig, axm = plt.subplots(figsize=(8, 7))
im = axm.imshow(_camb, cmap='RdBu_r', vmin=-_vmax, vmax=_vmax)
axm.set_title(f'Cambio de A: media {anios[_mitad]}-{anios[-1]} menos {anios[0]}-{anios[_mitad - 1]}',
              fontweight='bold')
axm.axis('off'); fig.colorbar(im, ax=axm, fraction=.046, pad=.04, label='Δ A (t/ha/año)')
plt.tight_layout(); plt.show()


In [ ]:
# Serie de A por punto: A_y(punto) = R_y(punto) · (K · LS · C · P constantes)
_pg = puntos_gdf.to_crs(CRS_GEOGRAFICO)
_coords = list(zip(_pg.geometry.x, _pg.geometry.y))
with rasterio.open(EE_R_ANUAL_FILE) as s:
    _bandas = list(range(1, len(anios) + 1))
    R_pts = np.array([list(s.sample([c], indexes=_bandas))[0] for c in _coords], dtype='float64')

_base_klcp = (puntos_gdf['K_factor'] * puntos_gdf['LS_oficial']
              * puntos_gdf['C_factor'] * puntos_gdf['P_factor']).values
A_pts_anual = R_pts * _base_klcp[:, None]
for j, y in enumerate(anios):
    puntos_gdf[f'A_{y}'] = A_pts_anual[:, j]
puntos_gdf['A_tendencia_Sen'] = [tendencia_mk_sen(anios, A_pts_anual[i])['sen_slope']
                                 for i in range(len(puntos_gdf))]

print('Tendencia de A por punto (pendiente de Sen, t/ha/año por año):')
display(puntos_gdf['A_tendencia_Sen'].describe().round(3))
print('Puntos con tendencia creciente:', int((puntos_gdf['A_tendencia_Sen'] > 0).sum()), '/ 117')


## 27. Síntesis: objetivo general y prioridades de conservación

Se integran los resultados de los cinco objetivos en un cuadro de síntesis y se
delimitan las **áreas prioritarias para la conservación de suelos**, definidas
como la intersección de **pérdida de suelo alta o muy alta** ($A > 50$ t/ha/año)
con **riesgo potencial alto o muy alto** (LS × R nivel 4–5).


In [ ]:
# Áreas prioritarias = pérdida alta/muy alta  Y  riesgo LSxR alto/muy alto
_prio = (mask_cuenca & np.isfinite(A_modelo) & (A_modelo > EROSION_BORDES[2])
         & np.isfinite(LSR_clase) & (LSR_clase >= 4))
prioridad = np.where(mask_cuenca, _prio.astype('float32'), np.nan).astype('float32')
guardar_raster(PRIORIDAD_FILE, prioridad, perfil_ls)

_ha_prio = float(np.nansum(prioridad == 1) * AREA_PX_HA)
_ha_cuenca = float(np.isfinite(prioridad).sum() * AREA_PX_HA)
print(f'Áreas prioritarias de conservación: {_ha_prio:,.0f} ha  '
      f'({100 * _ha_prio / _ha_cuenca:.1f}% de la zona de estudio).')

_idx_k = corr_k['r_Pearson'].abs().idxmax()
sintesis = pd.DataFrame([
    {'objetivo': 'I. Caracterización de los suelos',
     'resultado_clave': (f"n=117; texturas dominantes: "
                         f"{', '.join(puntos_gdf['Textura_USDA'].value_counts().head(3).index)}; "
                         f"arcilla media {puntos_gdf['Arcilla_pct'].mean():.0f}%, MO media {puntos_gdf['MO_pct'].mean():.1f}%")},
    {'objetivo': 'II. Factor K y su relación con las propiedades',
     'resultado_clave': (f"K medio {puntos_gdf['K_factor'].mean():.3f} (SI); "
                         f"correlación más fuerte con {_idx_k} (r={corr_k.loc[_idx_k, 'r_Pearson']:+.2f})")},
    {'objetivo': 'III. Riesgo potencial (LS × R)',
     'resultado_clave': (f"{resumen_riesgo.loc[resumen_riesgo['nivel'] >= 4, 'area_pct'].sum():.1f}% "
                         f"de la zona en riesgo alto/muy alto")},
    {'objetivo': 'IV. Pérdidas de suelo (USLE/RUSLE)',
     'resultado_clave': (f"A media {resumen['A_media_t_ha'].iloc[-1]:.1f} t/ha/año; "
                         f"pérdida total {resumen['perdida_t_ano'].iloc[-1] / 1e6:.2f} Mt/año; "
                         f"{resumen.loc[resumen['clase'].isin([3, 4]), 'area_pct'].sum():.1f}% en erosión alta/muy alta")},
    {'objetivo': 'V. Variación temporal 2010–2025',
     'resultado_clave': (f"tendencia de A: {mk_A['tendencia']} "
                         f"(Sen {mk_A['sen_slope']:+.2f} t/ha/año·año, p={mk_A['p_value']:.2f})")},
    {'objetivo': 'General. Prioridades de conservación',
     'resultado_clave': f"{_ha_prio:,.0f} ha prioritarias ({100 * _ha_prio / _ha_cuenca:.1f}% de la zona)"},
])
sintesis.to_csv(SINTESIS_CSV, index=False)
print('\n=== SÍNTESIS POR OBJETIVO ===')
display(sintesis)

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(np.where(mask_cuenca, 0.12, np.nan), cmap='Greys', vmin=0, vmax=1)
ax.imshow(np.where(prioridad == 1, 1, np.nan), cmap=ListedColormap(['#b2182b']))
ax.set_title('Áreas prioritarias para la conservación de suelos\n'
             '(pérdida alta/muy alta  ∩  riesgo LS×R alto/muy alto)', fontweight='bold')
ax.axis('off'); plt.tight_layout(); plt.show()


## 28. Exportación final — modelo USLE/RUSLE completo

Se reexporta la tabla de puntos con **todos** los factores y resultados de los
cinco objetivos (propiedades del suelo, K, C, P, R, LS×R, A por punto y por
ráster, clases y tendencia), junto con el inventario de rásteres y tablas.


In [ ]:
_ = exportar_resultados(puntos_gdf, ruta_xlsx=SALIDA_PUNTOS_XLSX, ruta_gpkg=SALIDA_PUNTOS_GPKG)

# --- Hoja de precipitación mensual media (climatología CHIRPS 2020–2025) por punto ---
#     Incluye MFI y Factor R para el análisis de correlación P-mensual ↔ R.
_cols_p = [f'P_{m:02d}' for m in range(1, 13)]
_id_cols = [c for c in ['ID_UNAL', 'ID_CIAT', 'LONG', 'LAT', COL_X_PROJ, COL_Y_PROJ, 'MSNM']
            if c in puntos_gdf.columns]
_prec = puntos_gdf[_id_cols + [c for c in _cols_p if c in puntos_gdf.columns]].copy()
_prec = _prec.rename(columns={f'P_{m:02d}': n for m, n in enumerate(
    ['P_Ene', 'P_Feb', 'P_Mar', 'P_Abr', 'P_May', 'P_Jun',
     'P_Jul', 'P_Ago', 'P_Sep', 'P_Oct', 'P_Nov', 'P_Dic'], start=1)})
for _c in ['P_anual', 'MFI', 'Factor_R']:
    if _c in puntos_gdf.columns:
        _prec[_c] = puntos_gdf[_c].values
_prec.to_csv(PRECIP_MENSUAL_CSV, index=False)

# xlsx multi-hoja: modelo + precipitación mensual
with pd.ExcelWriter(SALIDA_PUNTOS_XLSX, engine='openpyxl') as _xw:
    puntos_gdf.drop(columns='geometry', errors='ignore').to_excel(_xw, sheet_name='RUSLE', index=False)
    _prec.to_excel(_xw, sheet_name='Precipitacion_mensual', index=False)
print(f'Hoja "Precipitacion_mensual" ({_prec.shape[0]}×{_prec.shape[1]}) añadida a '
      f'{SALIDA_PUNTOS_XLSX.name}  y  {PRECIP_MENSUAL_CSV.name}')

# Escribir también esa hoja DENTRO de la base de la tesis (Cuenca Amaime Tesis Dayana.xlsx)
if AGREGAR_HOJA_A_BASE_SUELOS:
    try:
        with pd.ExcelWriter(ARCHIVO_SUELOS, engine='openpyxl', mode='a',
                            if_sheet_exists='replace') as _xw2:
            _prec.to_excel(_xw2, sheet_name='Precipitacion_mensual', index=False)
        print(f'Hoja "Precipitacion_mensual" añadida a {ARCHIVO_SUELOS.name}')
    except Exception as _e:
        print(f'AVISO: no se pudo escribir en {ARCHIVO_SUELOS.name} (ciérralo en Excel y '
              f'ejecuta scripts/agregar_hoja_precipitacion.py). Detalle: {_e}')

print('\n=== RÁSTERES GENERADOS (Salida/) ===')
for _r in [WORLDCOVER_FILE, FACTOR_K_FILE, FACTOR_C_FILE, FACTOR_P_FILE,
           R_CLIM_FILE, R_PROM_FILE, EE_R_ANUAL_FILE, RIESGO_LSR_FILE,
           A_CLIM_FILE, A_PROM_FILE, A_STACK_FILE, TREND_A_FILE, PRIORIDAD_FILE]:
    print(f'  [{"OK" if _r.exists() else "FALTA":5s}] {_r.name}')

print('\n=== TABLAS ===')
for _r in [SALIDA_PUNTOS_XLSX, SALIDA_PUNTOS_GPKG, CARACTERIZACION_CSV, CORR_K_CSV,
           RIESGO_LSR_CSV, RESUMEN_CLASES_CSV, SERIE_TEMP_CSV, SINTESIS_CSV]:
    print(f'  [{"OK" if _r.exists() else "FALTA":5s}] {_r.name}')

print('\n=== SÍNTESIS POR OBJETIVO ===')
display(sintesis)


## 29. Referencias

- **Arnoldus, H.M.J. (1980).** An approximation of the rainfall factor in the USLE. En *Assessment of Erosion*, Wiley.
- **ESA (2021).** *WorldCover 10 m 2021 v200.* European Space Agency. doi:10.5281/zenodo.7254221
- **European Space Agency / Airbus.** *Copernicus DEM GLO‑30* (Global 30 m DEM).
- **FAO‑PNUMA‑UNESCO (1980).** *Metodología provisional para la evaluación de la degradación de los suelos.* FAO, Roma.
- **Funk, C., et al. (2015).** The climate hazards infrared precipitation with stations (**CHIRPS**). *Scientific Data*, 2, 150066.
- **Lindsay, J.B. (2016).** Whitebox GAT / **WhiteboxTools**: an open-source tool for geomorphometric analysis. *Computers & Geosciences*, 95, 75–84.
- **Mitasova, H., Hofierka, J., Zlocha, M., Iverson, L.R. (1996).** Modelling topographic potential for erosion and deposition using GIS. *Int. J. GIS*, 10(5), 629–641. (Formulación **LS** de área de contribución; Mitasova & Mitas, 2001.)
- **Renard, K.G., Foster, G.R., Weesies, G.A., McCool, D.K., Yoder, D.C. (1997).** *Predicting Soil Erosion by Water: RUSLE.* USDA Agriculture Handbook 703.
- **Renard, K.G., Freimund, J.R. (1994).** Using monthly precipitation data to estimate the **R‑factor** in the revised USLE. *Journal of Hydrology*, 157, 287–306.
- **Wischmeier, W.H., Smith, D.D. (1978).** *Predicting Rainfall Erosion Losses: A Guide to Conservation Planning.* USDA Agriculture Handbook 537. (Ecuación del **factor K** y del factor L·S clásico.)
